<a href="https://colab.research.google.com/github/alaaguedda/esg-sustainability-reports/blob/main/esg_report_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip uninstall -y bitsandbytes -q
!pip install -q "bitsandbytes==0.45.5" --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.2 MB/s eta 0:00:00


In [3]:
# Stage 0.0: Environment bootstrap — installs pinned to versions known to work
# together on Colab Free T4 (CUDA 12.x) as of this project's start. Rationale
# for each package lives inline; nothing here is decorative.
#
# pdfplumber + pymupdf (fitz): redundant PDF backends. pymupdf is fast for
#   page-count checks (used during filtering, before we commit to full text
#   extraction); pdfplumber is used for the actual text extraction because it
#   preserves reading order / layout better for report-style PDFs.
# huggingface_hub: dataset file listing + streaming download without cloning
#   the whole repo (repo is large; we only want a filtered subset of PDFs).
# bitsandbytes + accelerate + transformers: 4-bit quantized local inference,
#   required to fit a 7B model in Colab Free's ~15GB T4 VRAM / 12GB RAM.
# sentence-transformers + scikit-learn + scipy: used later (Stage 3/5) for
#   embedding-based primary metrics and statistical testing; installed now so
#   the environment cell only needs to run once per session.
!pip install -q --no-deps \
    "transformers==4.46.3" \
    "accelerate==1.1.1" \
    "sentencepiece==0.2.0" \
    "pdfplumber==0.11.4" \
    "pymupdf==1.24.13" \
    "huggingface_hub==0.26.2" \
    "sentence-transformers==3.3.1" \
    "scikit-learn==1.5.2" \
    "scipy==1.13.1" \
    "pandas==2.2.2" \
    "rapidfuzz==3.10.1"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [4]:
!pip uninstall -y tokenizers -q
!pip install -q --no-deps "tokenizers==0.20.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.7 MB/s eta 0:00:00


In [5]:
!pip install -q --no-deps "pdfminer.six==20231228"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 74.6 MB/s eta 0:00:00


In [6]:
import transformers, sentence_transformers, sklearn, scipy, pandas, rapidfuzz, fitz, pdfplumber, huggingface_hub
print("all core imports OK")

all core imports OK


In [7]:
import subprocess
print(subprocess.run(["python", "-c", "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout)

torch 2.11.0+cu128 cuda True



In [8]:
import torch, torchvision
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("cuda", torch.version.cuda, "available:", torch.cuda.is_available())

torch 2.11.0+cu128
torchvision 0.26.0+cu128
cuda 12.8 available: True


In [9]:
# Stage 0.1: Mount Drive and materialize the fixed directory schema.
# Every persistent artifact in this project is written under PROJECT_ROOT so
# that a fresh Colab runtime (local disk wiped) can resume purely by
# re-mounting Drive and re-reading cached files — no recomputation of
# already-completed work.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = "/content/drive/MyDrive/esg_project"

DIR_SCHEMA = {
    "raw_pdfs_cache":     os.path.join(PROJECT_ROOT, "raw_pdfs_cache"),
    "extracted_text":     os.path.join(PROJECT_ROOT, "extracted_text"),
    "reference_profiles": os.path.join(PROJECT_ROOT, "reference_profiles"),
    "compressed_A":       os.path.join(PROJECT_ROOT, "compressed", "A"),
    "compressed_B":       os.path.join(PROJECT_ROOT, "compressed", "B"),
    "compressed_C":       os.path.join(PROJECT_ROOT, "compressed", "C"),
    "compressed_baseline":os.path.join(PROJECT_ROOT, "compressed", "baseline"),
    "expanded_A":         os.path.join(PROJECT_ROOT, "expanded", "A"),
    "expanded_B":         os.path.join(PROJECT_ROOT, "expanded", "B"),
    "expanded_C":         os.path.join(PROJECT_ROOT, "expanded", "C"),
    "expanded_baseline":  os.path.join(PROJECT_ROOT, "expanded", "baseline"),
    "evaluations":        os.path.join(PROJECT_ROOT, "evaluations"),
    "logs":               os.path.join(PROJECT_ROOT, "logs"),
    "config":             os.path.join(PROJECT_ROOT, "config"),
    "manifest":           os.path.join(PROJECT_ROOT, "manifest"),
    "gen_cache":          os.path.join(PROJECT_ROOT, "gen_cache"),
}

for _name, _path in DIR_SCHEMA.items():
    os.makedirs(_path, exist_ok=True)

print("Directory schema ready under:", PROJECT_ROOT)
for k, v in DIR_SCHEMA.items():
    print(f"  {k:22s} -> {v}")

Mounted at /content/drive
Directory schema ready under: /content/drive/MyDrive/esg_project
  raw_pdfs_cache         -> /content/drive/MyDrive/esg_project/raw_pdfs_cache
  extracted_text         -> /content/drive/MyDrive/esg_project/extracted_text
  reference_profiles     -> /content/drive/MyDrive/esg_project/reference_profiles
  compressed_A           -> /content/drive/MyDrive/esg_project/compressed/A
  compressed_B           -> /content/drive/MyDrive/esg_project/compressed/B
  compressed_C           -> /content/drive/MyDrive/esg_project/compressed/C
  compressed_baseline    -> /content/drive/MyDrive/esg_project/compressed/baseline
  expanded_A             -> /content/drive/MyDrive/esg_project/expanded/A
  expanded_B             -> /content/drive/MyDrive/esg_project/expanded/B
  expanded_C             -> /content/drive/MyDrive/esg_project/expanded/C
  expanded_baseline      -> /content/drive/MyDrive/esg_project/expanded/baseline
  evaluations            -> /content/drive/MyDrive/esg_pr

In [10]:
# Stage 0.2: Single central config. Every later stage reads exclusively from
# this dict (or the CONFIG object below) — no scattered magic numbers.
# This cell is intentionally the ONLY place experiment-level constants are
# defined; changing an experiment parameter means editing this cell and
# nothing else.
import json
import random
import numpy as np

CONFIG = {
    # --- Reproducibility ---
    "SEED": 42,

    # --- Model (single AI engine for all roles: profile / compression /
    # expansion / LLM-judge). Qwen2.5-7B-Instruct chosen over larger Qwen
    # variants because 4-bit NF4 quantization of the 7B checkpoint fits
    # comfortably in T4's ~15GB VRAM alongside KV-cache headroom for
    # ~10-page-report-length contexts; the 14B variant does not reliably fit
    # with room for generation on Colab Free.
    "MODEL_NAME": "Qwen/Qwen2.5-7B-Instruct",
    "QUANT_CONFIG": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "bfloat16",
        "bnb_4bit_use_double_quant": True,
    },
    "GENERATION_DEFAULTS": {
        "temperature": 0.0,      # deterministic; transformers requires do_sample=False
        "do_sample": False,
        "max_new_tokens": 2048,
        "repetition_penalty": 1.05,
    },

    # --- Dataset acquisition ---
    "HF_DATASET_REPO": "alaaguedda/esg-sustainability-reports",
    "HF_DATASET_REPO_TYPE": "dataset",
    "PDF_SUBDIR": "pdfs",

    # --- Stage 0 filtering / sampling ---
    "PAGE_COUNT_MIN": 10,
    "PAGE_COUNT_MAX": 120,
    "PAGE_COUNT_PREFERRED": 100,
    "MAX_REPORTS": 50,
    "MAX_REPORTS_PER_COMPANY": 3,  # caps over-representation of prolific companies
    "COMPANY_FOLDER_MAX_LEN": 30,
    "COMPANY_FOLDER_REJECT_TERMS": [
        "top", "list", "best", "report", "reports", "sustainability", "esg",
    ],

    # --- Compression ---
    "COMPRESSION_TARGET_RATIO": 0.10,   # fraction of original token count
    "COMPRESSION_TOLERANCE": 0.03,      # +/- absolute tolerance on ratio
    "COMPRESSION_MAX_REGEN_ATTEMPTS": 3,

    # --- Paths (mirrors DIR_SCHEMA; duplicated here so CONFIG is
    # self-contained and serializable independent of the Drive-mount cell) ---
    "PATHS": DIR_SCHEMA,
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

# Persist config to Drive so future sessions/conversations can diff against
# it and detect accidental parameter drift.
_config_path = os.path.join(DIR_SCHEMA["config"], "config.json")
with open(_config_path, "w") as f:
    json.dump({k: v for k, v in CONFIG.items()}, f, indent=2, default=str)

print(f"Config written to {_config_path}")
print(json.dumps({k: v for k, v in CONFIG.items() if k != "PATHS"}, indent=2))

Config written to /content/drive/MyDrive/esg_project/config/config.json
{
  "SEED": 42,
  "MODEL_NAME": "Qwen/Qwen2.5-7B-Instruct",
  "QUANT_CONFIG": {
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_use_double_quant": true
  },
  "GENERATION_DEFAULTS": {
    "temperature": 0.0,
    "do_sample": false,
    "max_new_tokens": 2048,
    "repetition_penalty": 1.05
  },
  "HF_DATASET_REPO": "alaaguedda/esg-sustainability-reports",
  "HF_DATASET_REPO_TYPE": "dataset",
  "PDF_SUBDIR": "pdfs",
  "PAGE_COUNT_MIN": 10,
  "PAGE_COUNT_MAX": 120,
  "PAGE_COUNT_PREFERRED": 100,
  "MAX_REPORTS": 50,
  "MAX_REPORTS_PER_COMPANY": 3,
  "COMPANY_FOLDER_MAX_LEN": 30,
  "COMPANY_FOLDER_REJECT_TERMS": [
    "top",
    "list",
    "best",
    "report",
    "reports",
    "sustainability",
    "esg"
  ],
  "COMPRESSION_TARGET_RATIO": 0.1,
  "COMPRESSION_TOLERANCE": 0.03,
  "COMPRESSION_MAX_REGEN_ATTEMPTS": 3
}


In [11]:
# Stage 0.3: Structured logging. All skip/reject/error events across every
# stage funnel through log_event() so the project has one auditable,
# machine-parseable log rather than scattered print statements. Appends to a
# JSONL file on Drive (survives session wipes) and mirrors to stdout.
import datetime
import threading

_log_lock = threading.Lock()
_LOG_PATH = os.path.join(CONFIG["PATHS"]["logs"], "pipeline_log.jsonl")


def log_event(stage: str, level: str, event: str, **fields):
    """Append a single structured log record.

    Parameters
    ----------
    stage : str
        Pipeline stage identifier, e.g. "stage0_filter", "stage2_compress_A".
    level : str
        One of "INFO", "WARN", "ERROR", "SKIP", "REJECT".
    event : str
        Short machine-stable event name, e.g. "company_rejected".
    **fields :
        Arbitrary JSON-serializable context (reasons, ids, metrics, etc).
    """
    record = {
        "ts": datetime.datetime.now(datetime.UTC).isoformat(),
        "stage": stage,
        "level": level,
        "event": event,
        **fields,
    }
    line = json.dumps(record, default=str)
    with _log_lock:
        with open(_LOG_PATH, "a") as f:
            f.write(line + "\n")
    print(f"[{record['ts']}] [{level}] [{stage}] {event} :: "
          f"{ {k: v for k, v in fields.items()} }")
    return record


log_event("bootstrap", "INFO", "logging_initialized", log_path=_LOG_PATH)

[2026-07-07T19:53:41.059524+00:00] [INFO] [bootstrap] logging_initialized :: {'log_path': '/content/drive/MyDrive/esg_project/logs/pipeline_log.jsonl'}


{'ts': '2026-07-07T19:53:41.059524+00:00',
 'stage': 'bootstrap',
 'level': 'INFO',
 'event': 'logging_initialized',
 'log_path': '/content/drive/MyDrive/esg_project/logs/pipeline_log.jsonl'}

In [12]:
# Stage 0.4: Generic content-addressed cache for expensive steps (extraction,
# profile generation, compression, expansion, evaluation, and raw LLM calls).
# Keyed by a hash of the *semantic inputs* of the call (not just a filename),
# so identical inputs always resolve to the same cache entry even across
# stages/conversations, and changed prompts/config automatically invalidate
# stale cache entries instead of silently reusing them.
import hashlib


def make_cache_key(*parts) -> str:
    """Deterministic hash over an arbitrary sequence of stringifiable parts.

    Every caller passes the exact set of inputs that determine the output
    (e.g. text content, prompt template id, model name, target ratio) so
    that a change to any of them naturally produces a new key.
    """
    h = hashlib.sha256()
    for p in parts:
        h.update(str(p).encode("utf-8"))
        h.update(b"\x1f")  # unit separator, avoids trivial collisions across parts
    return h.hexdigest()[:24]


def cache_path(cache_dir: str, key: str, ext: str = "json") -> str:
    return os.path.join(cache_dir, f"{key}.{ext}")


def cache_get_json(cache_dir: str, key: str):
    """Return cached JSON object, or None if not present / unreadable."""
    p = cache_path(cache_dir, key, "json")
    if not os.path.exists(p):
        return None
    try:
        with open(p, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        log_event("cache", "WARN", "cache_read_failed", path=p)
        return None


def cache_set_json(cache_dir: str, key: str, obj) -> str:
    """Write obj as JSON to the cache atomically (write-then-rename) so a
    Colab interruption mid-write never leaves a corrupt cache entry that
    would silently poison a later resume."""
    os.makedirs(cache_dir, exist_ok=True)
    p = cache_path(cache_dir, key, "json")
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, p)
    return p


def cache_get_text(cache_dir: str, key: str):
    p = cache_path(cache_dir, key, "txt")
    if not os.path.exists(p):
        return None
    with open(p, "r", encoding="utf-8") as f:
        return f.read()


def cache_set_text(cache_dir: str, key: str, text: str) -> str:
    os.makedirs(cache_dir, exist_ok=True)
    p = cache_path(cache_dir, key, "txt")
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, p)
    return p


log_event("bootstrap", "INFO", "cache_utils_ready", gen_cache_dir=CONFIG["PATHS"]["gen_cache"])

[2026-07-07T19:53:44.774333+00:00] [INFO] [bootstrap] cache_utils_ready :: {'gen_cache_dir': '/content/drive/MyDrive/esg_project/gen_cache'}


{'ts': '2026-07-07T19:53:44.774333+00:00',
 'stage': 'bootstrap',
 'level': 'INFO',
 'event': 'cache_utils_ready',
 'gen_cache_dir': '/content/drive/MyDrive/esg_project/gen_cache'}

In [23]:
# Stage 0.5: Load the single AI engine once per session, reused by every
# stage (profile generation, compression, expansion, LLM-judge evaluation).
# Deterministic generation (do_sample=False, greedy) + fixed seed satisfy the
# project's determinism requirement; note that "temperature=0" for an
# autoregressive HF model is implemented as greedy decoding (do_sample=False)
# since temperature itself is only meaningful under sampling.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import set_seed as hf_set_seed

hf_set_seed(CONFIG["SEED"])

_MODEL = None
_TOKENIZER = None


def get_model_and_tokenizer():
    """Lazily load and cache the model/tokenizer as module-level singletons
    so repeated calls across Stage 1-6 cells do not reload the 7B checkpoint.
    """
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=CONFIG["QUANT_CONFIG"]["load_in_4bit"],
        bnb_4bit_quant_type=CONFIG["QUANT_CONFIG"]["bnb_4bit_quant_type"],
        bnb_4bit_compute_dtype=getattr(torch, CONFIG["QUANT_CONFIG"]["bnb_4bit_compute_dtype"]),
        bnb_4bit_use_double_quant=CONFIG["QUANT_CONFIG"]["bnb_4bit_use_double_quant"],
    )

    log_event("model_load", "INFO", "loading_model", model=CONFIG["MODEL_NAME"])

    _TOKENIZER = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
    _MODEL = AutoModelForCausalLM.from_pretrained(
        CONFIG["MODEL_NAME"],
        quantization_config=quant_cfg,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    _MODEL.eval()

    log_event("model_load", "INFO", "model_ready",
              device=str(_MODEL.device) if hasattr(_MODEL, "device") else "sharded")
    return _MODEL, _TOKENIZER


def generate(prompt: str, system: str = None, max_new_tokens: int = None,
             cache_dir: str = None, cache_extra: str = "") -> str:
    gen_kwargs = dict(CONFIG["GENERATION_DEFAULTS"])
    if max_new_tokens is not None:
        gen_kwargs["max_new_tokens"] = max_new_tokens

    key = None
    if cache_dir is not None:
        key = make_cache_key(CONFIG["MODEL_NAME"], system or "", prompt,
                              json.dumps(gen_kwargs, sort_keys=True), cache_extra)
        cached = cache_get_text(cache_dir, key)
        if cached is not None:
            return cached

    model, tok = get_model_and_tokenizer()
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    input_ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            input_ids,
            do_sample=gen_kwargs["do_sample"],
            max_new_tokens=gen_kwargs["max_new_tokens"],
            repetition_penalty=gen_kwargs["repetition_penalty"],
            pad_token_id=tok.eos_token_id,
        )
    new_tokens = out_ids[0][input_ids.shape[1]:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # explicit cleanup — free large tensors immediately instead of waiting
    # for Python's garbage collector, so the allocator can reclaim/reuse
    # the memory before the next generate() call
    del input_ids, out_ids, new_tokens
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    if cache_dir is not None:
        cache_set_text(cache_dir, key, text)

    return text


def count_tokens(text: str) -> int:
    """Canonical token-counting function — used everywhere a report's size
    or a compression ratio needs to be measured, so every stage agrees on
    what a 'token' is (the target model's own tokenizer, not a proxy like
    whitespace splitting)."""
    _, tok = get_model_and_tokenizer()
    return len(tok(text, add_special_tokens=False)["input_ids"])

In [24]:
# Stage 0.6: List and filter the HF dataset's company folders WITHOUT
# downloading any PDFs yet. This is a metadata-only pass so filtering is
# cheap and fully re-runnable; only folders/files surviving the filter are
# ever downloaded (Stage 0.7+).
from huggingface_hub import HfApi
import re

_api = HfApi()


def list_pdf_repo_files():
    """Return all repo-relative paths under pdfs/ in the HF dataset repo."""
    all_files = _api.list_repo_files(
        repo_id=CONFIG["HF_DATASET_REPO"],
        repo_type=CONFIG["HF_DATASET_REPO_TYPE"],
    )
    prefix = CONFIG["PDF_SUBDIR"] + "/"
    return [f for f in all_files if f.startswith(prefix) and f.lower().endswith(".pdf")]


def is_valid_company_folder(folder: str) -> (bool, str):
    """Apply the explicit company-folder heuristic (all rules must pass).
    Returns (is_valid, reason) so every decision is logged, not just failures.
    """
    if not folder:
        return False, "empty_folder_name"
    if not folder.isalpha():
        # single alphabetic token: letters only, no digits/underscores/spaces
        return False, "not_single_alphabetic_token"
    if len(folder) > CONFIG["COMPANY_FOLDER_MAX_LEN"]:
        return False, f"exceeds_max_len_{CONFIG['COMPANY_FOLDER_MAX_LEN']}"
    lower = folder.lower()
    for term in CONFIG["COMPANY_FOLDER_REJECT_TERMS"]:
        if term in lower:
            return False, f"contains_reject_term:{term}"
    return True, "ok"


def build_filtered_file_index():
    """Group repo PDF paths by company folder and apply the folder filter.
    Every rejected folder is logged exactly once with its reason.
    Returns dict: company -> list of repo-relative pdf paths.
    """
    raw_files = list_pdf_repo_files()
    log_event("stage0_filter", "INFO", "repo_pdf_listing_complete", count=len(raw_files))

    by_company = {}
    for path in raw_files:
        # expected shape: pdfs/<company>/<filename>.pdf
        parts = path.split("/")
        if len(parts) != 3:
            log_event("stage0_filter", "SKIP", "unexpected_path_shape", path=path)
            continue
        _, company, filename = parts
        by_company.setdefault(company, []).append(path)

    seen_folders = set()
    filtered = {}
    for company, paths in by_company.items():
        if company in seen_folders:
            continue
        seen_folders.add(company)
        ok, reason = is_valid_company_folder(company)
        if ok:
            filtered[company] = paths
        else:
            log_event("stage0_filter", "REJECT", "company_folder_rejected",
                       folder=company, reason=reason, n_files=len(paths))

    log_event("stage0_filter", "INFO", "folder_filter_complete",
              accepted=len(filtered), rejected=len(seen_folders) - len(filtered))
    return filtered


FILTERED_FILE_INDEX = build_filtered_file_index()
print(f"Accepted companies: {len(FILTERED_FILE_INDEX)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[2026-07-07T20:17:34.787580+00:00] [INFO] [stage0_filter] repo_pdf_listing_complete :: {'count': 2331}
[2026-07-07T20:17:34.793738+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '10_mining_startups_to_watch_in_2026_startus_insights', 'reason': 'not_single_alphabetic_token', 'n_files': 7}
[2026-07-07T20:17:34.796923+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '10_sustainable_logistics_companies_[2026]_startus_insights', 'reason': 'not_single_alphabetic_token', 'n_files': 10}
[2026-07-07T20:17:34.800657+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '2024_esg_082125_v26_cw_-_intuitive.com', 'reason': 'not_single_alphabetic_token', 'n_files': 1}
[2026-07-07T20:17:34.804659+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '20_unique_transportation_startups_you_should_know_in_2025', 'reason': 'not_single_alphabetic_token', 'n_files': 1}
[2026-07-07T20:17:34.808279+00:00] [REJECT] [stage0_filter] com

In [25]:
# Stage 0.7: Download candidate PDFs (only from accepted companies),
# check page count locally, and select up to MAX_REPORTS reports meeting the
# 10-15 page window (preferring ~10 pages), capped per company, with a fixed
# seed for the random component. Downloaded PDFs are cached to
# raw_pdfs_cache/ on Drive so re-running after an interruption does not
# re-download files already fetched.
import fitz  # pymupdf, used here only for fast page counts
from huggingface_hub import hf_hub_download

MANIFEST_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage0_manifest.json")


def _local_pdf_cache_path(repo_path: str) -> str:
    safe_name = repo_path.replace("/", "__")
    return os.path.join(CONFIG["PATHS"]["raw_pdfs_cache"], safe_name)


def ensure_pdf_downloaded(repo_path: str) -> str:
    """Download a single PDF from the HF dataset into the Drive-backed raw
    cache if not already present; return the local path either way."""
    local_path = _local_pdf_cache_path(repo_path)
    if os.path.exists(local_path):
        return local_path
    downloaded = hf_hub_download(
        repo_id=CONFIG["HF_DATASET_REPO"],
        repo_type=CONFIG["HF_DATASET_REPO_TYPE"],
        filename=repo_path,
    )
    with open(downloaded, "rb") as src, open(local_path, "wb") as dst:
        dst.write(src.read())
    return local_path


def get_page_count(local_pdf_path: str):
    try:
        with fitz.open(local_pdf_path) as doc:
            return doc.page_count
    except Exception as e:
        log_event("stage0_select", "ERROR", "pdf_open_failed",
                   path=local_pdf_path, error=str(e))
        return None


def select_reports():
    """Build the Stage 0 manifest: up to MAX_REPORTS reports in the target
    page-count window, capped per company, sampled with a fixed seed among
    qualifying reports when more candidates qualify than needed.

    Resumable: if a manifest already exists on Drive, it is loaded and
    returned as-is rather than recomputed (Stage 0 selection is treated as
    locked once produced, matching the "reference profile is fixed ground
    truth" philosophy applied one stage earlier).
    """
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
        log_event("stage0_select", "INFO", "manifest_loaded_from_cache",
                   n_reports=len(manifest["reports"]))
        return manifest

    rng = random.Random(CONFIG["SEED"])
    companies = sorted(FILTERED_FILE_INDEX.keys())  # sort for determinism before shuffle
    rng.shuffle(companies)

    qualifying = []  # list of dicts, evaluated lazily company by company
    per_company_count = {}

    for company in companies:
        if len(qualifying) >= CONFIG["MAX_REPORTS"] * 3:
            # enough candidates gathered to make a well-supported final
            # selection without scanning the entire (large) repo
            break
        paths = list(FILTERED_FILE_INDEX[company])
        rng.shuffle(paths)
        for repo_path in paths:
            if per_company_count.get(company, 0) >= CONFIG["MAX_REPORTS_PER_COMPANY"]:
                break
            local_path = ensure_pdf_downloaded(repo_path)
            n_pages = get_page_count(local_path)
            if n_pages is None:
                continue
            if not (CONFIG["PAGE_COUNT_MIN"] <= n_pages <= CONFIG["PAGE_COUNT_MAX"]):
                log_event("stage0_select", "SKIP", "page_count_out_of_range",
                           repo_path=repo_path, n_pages=n_pages)
                continue
            qualifying.append({
                "company": company,
                "repo_path": repo_path,
                "local_pdf_path": local_path,
                "n_pages": n_pages,
                "dist_from_preferred": abs(n_pages - CONFIG["PAGE_COUNT_PREFERRED"]),
            })
            per_company_count[company] = per_company_count.get(company, 0) + 1

    log_event("stage0_select", "INFO", "qualifying_candidates_found", n=len(qualifying))

    # Selection method: prefer reports closest to the preferred page count;
    # break ties with the fixed-seed shuffle order already applied above.
    # This is a deterministic, documented sampling method as required.
    qualifying.sort(key=lambda r: r["dist_from_preferred"])
    selected = qualifying[:CONFIG["MAX_REPORTS"]]

    for r in selected:
        r["report_id"] = make_cache_key(r["company"], r["repo_path"])

    manifest = {
        "seed": CONFIG["SEED"],
        "sampling_method": (
            "shuffle companies (fixed seed) -> per-company cap "
            f"({CONFIG['MAX_REPORTS_PER_COMPANY']}) -> filter by page range "
            f"[{CONFIG['PAGE_COUNT_MIN']},{CONFIG['PAGE_COUNT_MAX']}] -> "
            f"rank by |pages - {CONFIG['PAGE_COUNT_PREFERRED']}| -> "
            f"take top {CONFIG['MAX_REPORTS']}"
        ),
        "n_qualifying_total": len(qualifying),
        "n_selected": len(selected),
        "target_max_reports": CONFIG["MAX_REPORTS"],
        "reports": selected,
    }

    if len(selected) < CONFIG["MAX_REPORTS"]:
        log_event("stage0_select", "WARN", "fewer_than_target_reports",
                   found=len(selected), target=CONFIG["MAX_REPORTS"])

    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    log_event("stage0_select", "INFO", "manifest_written",
              path=MANIFEST_PATH, n_reports=len(selected))
    return manifest


STAGE0_MANIFEST = select_reports()
print(f"Selected {STAGE0_MANIFEST['n_selected']} reports "
      f"(of {STAGE0_MANIFEST['n_qualifying_total']} qualifying candidates).")

[2026-07-07T20:17:37.926627+00:00] [INFO] [stage0_select] manifest_loaded_from_cache :: {'n_reports': 50}
Selected 50 reports (of 118 qualifying candidates).


In [26]:
# @title
# Stage 0.8: Text extraction for every selected report, cached to Drive keyed
# by report_id (company + filename hash from the manifest) so re-extraction
# is skipped on resume. Uses pdfplumber for layout-aware extraction.
import pdfplumber


def extract_text_from_pdf(local_pdf_path: str) -> str:
    pages_text = []
    with pdfplumber.open(local_pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text() or ""
            pages_text.append(t)
    return "\n\n".join(pages_text).strip()


def extracted_text_path(report_id: str) -> str:
    return os.path.join(CONFIG["PATHS"]["extracted_text"], f"{report_id}.txt")


def get_or_extract_text(report: dict) -> str:
    path = extracted_text_path(report["report_id"])
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    text = extract_text_from_pdf(report["local_pdf_path"])
    if not text:
        log_event("stage0_extract", "WARN", "empty_extraction",
                   report_id=report["report_id"], repo_path=report["repo_path"])
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)
    return text


def run_stage0_extraction(manifest: dict) -> dict:
    """Extract (or load cached) text for every report in the manifest;
    attach n_tokens (canonical tokenizer-based count) to each record and
    persist the enriched manifest so downstream stages have token counts
    available without recomputation."""
    enriched = []
    for report in manifest["reports"]:
        text = get_or_extract_text(report)
        n_tokens = count_tokens(text) if text else 0
        enriched_report = dict(report)
        enriched_report["extracted_text_path"] = extracted_text_path(report["report_id"])
        enriched_report["n_chars"] = len(text)
        enriched_report["n_tokens_original"] = n_tokens
        enriched.append(enriched_report)
        log_event("stage0_extract", "INFO", "extraction_ready",
                   report_id=report["report_id"], n_pages=report["n_pages"],
                   n_tokens=n_tokens)

    manifest["reports"] = enriched
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest


STAGE0_MANIFEST = run_stage0_extraction(STAGE0_MANIFEST)

import pandas as pd
_manifest_df = pd.DataFrame(STAGE0_MANIFEST["reports"])
_manifest_csv_path = os.path.join(CONFIG["PATHS"]["manifest"], "stage0_manifest.csv")
_manifest_df.to_csv(_manifest_csv_path, index=False)
print(f"Stage 0 complete: {len(_manifest_df)} reports extracted.")
print(f"Manifest CSV: {_manifest_csv_path}")
_manifest_df[["company", "n_pages", "n_tokens_original"]].describe(include="all")

[2026-07-07T20:17:39.925526+00:00] [INFO] [model_load] loading_model :: {'model': 'Qwen/Qwen2.5-7B-Instruct'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[2026-07-07T20:24:41.043105+00:00] [INFO] [model_load] model_ready :: {'device': 'cuda:0'}
[2026-07-07T20:24:41.069280+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': 'bfd1b3b0ebe8410f3424243f', 'n_pages': 10, 'n_tokens': 3246}
[2026-07-07T20:24:41.425040+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '852815f0465c6a55e522d6cf', 'n_pages': 10, 'n_tokens': 8928}
[2026-07-07T20:24:41.754903+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '5e6d98d4006f9eec322846dd', 'n_pages': 10, 'n_tokens': 3485}
[2026-07-07T20:24:42.063957+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': 'bf96fa1c7ff813f490e79b07', 'n_pages': 10, 'n_tokens': 2631}
[2026-07-07T20:24:42.485482+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '35ed2dd1c7889989991747ee', 'n_pages': 10, 'n_tokens': 4228}
[2026-07-07T20:24:42.806088+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '4772a1d980f05b15994e318d', 'n_pages': 11, 'n_

,company,n_pages,n_tokens_original
count,50,50.00000,50.000000
unique,33,NaN,NaN
top,cropx,NaN,NaN
freq,3,NaN,NaN
mean,NaN,19.16000,8459.180000
std,NaN,7.12386,6243.121935
min,NaN,10.00000,1710.000000
25%,NaN,12.25000,4225.750000
50%,NaN,17.50000,6547.500000
75%,NaN,24.75000,10983.750000


In [27]:
# Stage 1.0 v2: Flattened Reference Profile schema. v1 (nested per-item
# objects for kpis/financial_metrics/environmental_metrics/targets/
# dates_events/numerical_values) is DEPRECATED but NOT deleted — profiles
# already generated under v1 remain valid ground truth and are read via the
# version-aware adapter (next cell), not regenerated. v2 is used for all
# NEW profile generation going forward.
#
# Change vs v1: every field is now `array of string` (or plain string).
# Structured facts that used to be sub-objects are now single, densely
# formatted strings, e.g.:
#   kpi:               "Scope 1 emissions: 98,400 tCO2e (FY2024)"
#   financial_metric:  "Revenue: 4.2B USD"
#   environmental_metric: "Water withdrawal: 1.1M m3"
#   target:            "Net zero by 2040 (target: net zero, year: 2040)"
#   date_event:        "2024-03-01: Published annual sustainability report"
#   numerical_value:   "18% (context: YoY Scope 1 reduction)"
# This removes the nested-object formatting burden that was the primary
# driver of empty arrays (see conversation analysis), while keeping every
# fact machine-parseable downstream via regex/fuzzy matching (Stage 3/5/6
# metrics already fuzzy-match on substrings, so a flat "name: value unit"
# string is equally usable as a matching target — often MORE robust, since
# it removes brittle exact key-based lookups).
REFERENCE_PROFILE_SCHEMA_VERSION_V2 = "2.0"

REFERENCE_PROFILE_SCHEMA_V2 = {
    "schema_version": REFERENCE_PROFILE_SCHEMA_VERSION_V2,
    "type": "object",
    "required": [
        "company", "year", "industry", "industry_inference_method",
        "executive_summary", "esg_topics", "entities", "organizations",
        "stakeholders", "numerical_values", "kpis", "financial_metrics",
        "environmental_metrics", "governance_information", "social_information",
        "risks", "opportunities", "commitments", "targets", "future_plans",
        "regulations_frameworks_standards", "dates_events",
    ],
    "properties": {
        "company": {"type": "string"},
        "year": {"type": ["string", "null"]},
        "industry": {"type": "string"},
        "industry_inference_method": {"type": "string"},
        "executive_summary": {"type": "string"},
        # Every field below: flat array of pre-formatted strings.
        **{
            f: {"type": "array", "items": {"type": "string"}}
            for f in [
                "esg_topics", "entities", "organizations", "stakeholders",
                "numerical_values", "kpis", "financial_metrics",
                "environmental_metrics", "governance_information",
                "social_information", "risks", "opportunities", "commitments",
                "targets", "future_plans", "regulations_frameworks_standards",
                "dates_events",
            ]
        },
    },
}

# Keep the v1 schema object (from the original Stage 1.0 cell) resident
# under its own name so the adapter cell below can still validate/read
# already-generated v1 profiles by version, rather than silently guessing
# their shape.
#REFERENCE_PROFILE_SCHEMA_V1 = REFERENCE_PROFILE_SCHEMA  # alias to the original
                                                          # nested schema object

# Going forward, REFERENCE_PROFILE_SCHEMA / REFERENCE_PROFILE_SCHEMA_VERSION
# point at v2 — every NEW generation call uses this.
REFERENCE_PROFILE_SCHEMA = REFERENCE_PROFILE_SCHEMA_V2
REFERENCE_PROFILE_SCHEMA_VERSION = REFERENCE_PROFILE_SCHEMA_VERSION_V2

_schema_v2_path = os.path.join(CONFIG["PATHS"]["config"], "reference_profile_schema_v2.json")
with open(_schema_v2_path, "w") as f:
    json.dump(REFERENCE_PROFILE_SCHEMA_V2, f, indent=2)
log_event("stage1_schema", "INFO", "schema_v2_locked_and_written",
          version=REFERENCE_PROFILE_SCHEMA_VERSION_V2, path=_schema_v2_path)

[2026-07-07T20:41:47.697506+00:00] [INFO] [stage1_schema] schema_v2_locked_and_written :: {'version': '2.0', 'path': '/content/drive/MyDrive/esg_project/config/reference_profile_schema_v2.json'}


{'ts': '2026-07-07T20:41:47.697506+00:00',
 'stage': 'stage1_schema',
 'level': 'INFO',
 'event': 'schema_v2_locked_and_written',
 'version': '2.0',
 'path': '/content/drive/MyDrive/esg_project/config/reference_profile_schema_v2.json'}

In [28]:
# Stage 1.0b: Sample a fixed subset of 5 reports from the Stage 0 manifest
# for this study run, instead of processing all 50. Deterministic (fixed
# seed) and cached to disk so re-running the notebook reuses the same 5
# reports rather than resampling.
STAGE1_SAMPLE_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage1_sample.json")
N_SAMPLE = 5

def get_stage1_sample(manifest: dict, n: int = N_SAMPLE) -> dict:
    if os.path.exists(STAGE1_SAMPLE_PATH):
        with open(STAGE1_SAMPLE_PATH, "r") as f:
            sample = json.load(f)
        log_event("stage1_sample", "INFO", "sample_loaded_from_cache",
                   n_reports=len(sample["reports"]))
        return sample

    rng = random.Random(CONFIG["SEED"])
    reports = list(manifest["reports"])
    rng.shuffle(reports)
    selected = reports[:n]

    sample = {"seed": CONFIG["SEED"], "n_sample": len(selected), "reports": selected}
    with open(STAGE1_SAMPLE_PATH, "w") as f:
        json.dump(sample, f, indent=2, default=str)
    log_event("stage1_sample", "INFO", "sample_written",
               path=STAGE1_SAMPLE_PATH, n_reports=len(selected))
    return sample


STAGE1_SAMPLE = get_stage1_sample(STAGE0_MANIFEST, N_SAMPLE)
print(f"Sampled {STAGE1_SAMPLE['n_sample']} reports for Stage 1: "
      f"{[r['report_id'] for r in STAGE1_SAMPLE['reports']]}")

[2026-07-07T20:41:50.798072+00:00] [INFO] [stage1_sample] sample_loaded_from_cache :: {'n_reports': 5}
Sampled 5 reports for Stage 1: ['a9a45ce8faeb53d10d8d0dd3', '8777dcf4b50e5d00da6e73de', '26aef234bc0eb9665b42be0d', '15f5ef3cd925ac417b95388c', '35ed2dd1c7889989991747ee']


In [ ]:
# check report sizes in your sample before generating
for r in STAGE1_SAMPLE["reports"]:
    print(r["report_id"], r.get("n_tokens_original"))

a9a45ce8faeb53d10d8d0dd3 5674
8777dcf4b50e5d00da6e73de 7770
26aef234bc0eb9665b42be0d 8162
15f5ef3cd925ac417b95388c 6496
35ed2dd1c7889989991747ee 4228


In [ ]:
print(f"Model memory footprint: {_MODEL.get_memory_footprint() / 1e9:.2f} GB")

# check if any layers are actually 4-bit
import bitsandbytes as bnb
n_4bit, n_other = 0, 0
for name, module in _MODEL.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        n_4bit += 1
    elif isinstance(module, torch.nn.Linear):
        n_other += 1
print("Linear4bit layers:", n_4bit, "| plain Linear layers:", n_other)

Model memory footprint: 5.44 GB
Linear4bit layers: 196 | plain Linear layers: 1


In [ ]:
print(_MODEL.hf_device_map)
print(_MODEL.config.quantization_config)

{'': 0}
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [29]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(torch.cuda.memory_allocated()/1e9, "GB allocated")
print(torch.cuda.memory_reserved()/1e9, "GB reserved")

5.5471232 GB allocated
5.584715776 GB reserved


In [30]:
# Stage 1.1 v2: Reference Profile generation — flat schema, greedier prompt,
# coerce-before-validate (fixes type mismatches locally so the RETRY LOOP is
# reserved for genuine JSON-parse failures, not trivial type coercion —
# eliminating the wasted regeneration calls seen in the logs), and a
# version-aware read adapter so v1 (nested) profiles already on Drive keep
# working in every downstream stage without regeneration.
import gc
import jsonschema
try:
    import jsonschema  # noqa: F811
except ImportError:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "jsonschema==4.23.0"])
    import jsonschema  # noqa: F811

REFERENCE_PROFILE_PROMPT_VERSION = "v2"
MAX_PROFILE_INPUT_TOKENS = 4000
PROFILE_MAX_NEW_TOKENS = 3072

# Rebalanced system prompt: explicit permission (even instruction) to be
# inclusive/generous, paired with a SINGLE narrow anti-hallucination
# constraint (don't invent facts that aren't in the text at all) instead of
# many overlapping "be exact" phrasings that previously stacked up into an
# overall "when unsure, omit" bias.
_PROFILE_SYSTEM_PROMPT = (
    "You are an ESG/sustainability report analyst. Your job is to extract "
    "AS MUCH relevant information as you can find in the report into the "
    "given JSON fields. Be GENEROUS and INCLUSIVE: if a fact is present in "
    "the text and plausibly belongs in a field, PUT IT THERE even if you "
    "are not 100% sure it is the perfect category — a reasonable categorization "
    "is far more useful than leaving a field empty. The only hard rule is: "
    "do not invent facts, numbers, or names that are not present anywhere in "
    "the source text. Missing a real fact from the report is a WORSE mistake "
    "than placing a real fact in a slightly imperfect field. "
    "Output ONLY a single valid JSON object matching the schema. No prose, "
    "no markdown code fences, no commentary before or after the JSON."
)

# One compact positive example demonstrating DENSITY (several items per
# array, flat pre-formatted strings) rather than only describing the format
# in prose — small instruction-tuned models imitate demonstrated density
# far more reliably than they follow abstract "be thorough" instructions.
_PROFILE_FEW_SHOT_EXAMPLE = """EXAMPLE (abbreviated report -> abbreviated profile):
REPORT EXCERPT: "GreenCorp cut Scope 1 emissions 18% YoY (120,000->98,400 tCO2e,
2023-2024) via renewable electricity at its 3 largest plants. The Audit
Committee met 4 times in FY2024 and launched a whistleblower hotline (42
reports, 12 substantiated). GreenCorp targets net zero by 2040 and follows
the GRI Standards."

EXAMPLE PROFILE (partial):
{
  "esg_topics": ["emissions reduction", "renewable electricity", "whistleblower protection"],
  "kpis": ["Scope 1 emissions: 98,400 tCO2e (2024)", "Scope 1 emissions: 120,000 tCO2e (2023)"],
  "governance_information": ["Audit Committee met 4 times in FY2024", "Whistleblower hotline: 42 reports, 12 substantiated"],
  "targets": ["Net zero by 2040"],
  "regulations_frameworks_standards": ["GRI Standards"]
}
Notice EVERY concrete fact in the excerpt appears somewhere above — that is the density expected from you."""


def _truncate_for_profile_input(report_text: str, max_tokens: int = MAX_PROFILE_INPUT_TOKENS):
    _, tok = get_model_and_tokenizer()
    ids = tok(report_text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return report_text, False
    truncated_ids = ids[:max_tokens]
    return tok.decode(truncated_ids, skip_special_tokens=True), True


def _build_profile_prompt(report_text: str) -> str:
    schema_keys = ", ".join(REFERENCE_PROFILE_SCHEMA["required"])
    return f"""{_PROFILE_FEW_SHOT_EXAMPLE}

Now extract a Reference Profile from the REAL report below.

Return a single JSON object with EXACTLY these top-level keys (no more, no fewer):
{schema_keys}

Format rules:
- "company", "year", "industry", "industry_inference_method", "executive_summary": plain strings.
- ALL other fields are arrays of plain strings (no nested objects, no sub-keys).
- For numeric/structured facts, pack the detail into ONE readable string, e.g.:
  kpis: "Scope 1 emissions: 98,400 tCO2e (FY2024)"
  financial_metrics: "Revenue: 4.2B USD (FY2024)"
  environmental_metrics: "Water withdrawal: 1.1M m3"
  targets: "Net zero by 2040"
  dates_events: "2024-03-01: Published annual sustainability report"
  numerical_values: "18% YoY reduction in Scope 1 emissions"
- List EVERY distinct instance you find, deduplicated. Aim for completeness over caution.
- "industry_inference_method": one short phrase, e.g. "stated explicitly" or
  "inferred from described products/services".
- Do not invent facts absent from the text below, but DO include every real fact
  you find, even if you must place it in the closest-fitting field.

REPORT TEXT:
\"\"\"
{report_text}
\"\"\"

Return ONLY the JSON object."""


def _extract_json_object(raw_text: str) -> dict:
    text = raw_text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    start = text.find("{")
    if start == -1:
        raise ValueError("no_json_object_found")
    candidate = text[start:]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Truncated output recovery: close any unbalanced brackets/braces and
    # strip a dangling trailing comma/partial value, then retry once.
    # This turns "generation hit max_new_tokens mid-array" into a usable
    # partial profile instead of a hard failure.
    depth_curly, depth_square = 0, 0
    in_string, escape = False, False
    last_safe_idx = 0
    for i, ch in enumerate(candidate):
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth_curly += 1
        elif ch == "}":
            depth_curly -= 1
        elif ch == "[":
            depth_square += 1
        elif ch == "]":
            depth_square -= 1
        if ch in ",}]" and depth_curly >= 0:
            last_safe_idx = i + 1

    repaired = candidate[:last_safe_idx].rstrip().rstrip(",")
    repaired += "]" * max(depth_square, 0) + "}" * max(depth_curly, 0)
    return json.loads(repaired)


def profile_path(report_id: str) -> str:
    return os.path.join(CONFIG["PATHS"]["reference_profiles"], f"{report_id}.json")


def _free_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _coerce_to_schema_v2(obj: dict) -> dict:
    """Fix common type mismatches locally (no model call) BEFORE validation,
    so schema-validation retries are reserved for genuinely unparsable
    output, not trivial coercions like None->"" or int year->str. This is
    exactly the class of retry seen in the logs (`None is not of type
    'string'`, `2022 is not of type 'string'`) — both fixed here for free.
    """
    scalar_string_fields = ["company", "industry", "industry_inference_method",
                             "executive_summary"]
    for f in scalar_string_fields:
        v = obj.get(f)
        if v is None:
            obj[f] = ""
        elif not isinstance(v, str):
            obj[f] = str(v)

    y = obj.get("year")
    if y is not None and not isinstance(y, str):
        obj["year"] = str(y)

    flat_array_fields = [
        "esg_topics", "entities", "organizations", "stakeholders",
        "numerical_values", "kpis", "financial_metrics", "environmental_metrics",
        "governance_information", "social_information", "risks", "opportunities",
        "commitments", "targets", "future_plans", "regulations_frameworks_standards",
        "dates_events",
    ]
    for f in flat_array_fields:
        items = obj.get(f, [])
        if not isinstance(items, list):
            items = [items] if items else []
        fixed = []
        for item in items:
            if isinstance(item, str):
                if item.strip():
                    fixed.append(item.strip())
            elif isinstance(item, dict):
                # Legacy nested-object-shaped item slipped through despite
                # the flat prompt: flatten it into one readable string
                # instead of discarding the fact it represents.
                parts = [f"{k}: {v}" for k, v in item.items() if v not in (None, "")]
                if parts:
                    fixed.append(", ".join(parts))
            elif item not in (None, ""):
                fixed.append(str(item))
        obj[f] = fixed

    # Drop any unexpected extra keys the model may have added, and ensure
    # every required key is present at all (missing key, not just wrong
    # type, still needs to raise so the retry loop catches truly broken
    # output rather than silently proceeding with a partial profile).
    return obj


def generate_reference_profile(report: dict) -> dict:
    out_path = profile_path(report["report_id"])
    if os.path.exists(out_path):
        with open(out_path, "r") as f:
            return json.load(f)

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        report_text = f.read()

    report_text, was_truncated = _truncate_for_profile_input(report_text)
    if was_truncated:
        log_event("stage1_profile", "WARN", "input_truncated_for_memory",
                   report_id=report["report_id"], max_tokens=MAX_PROFILE_INPUT_TOKENS)

    prompt = _build_profile_prompt(report_text)
    raw = generate(
        prompt,
        system=_PROFILE_SYSTEM_PROMPT,
        max_new_tokens=PROFILE_MAX_NEW_TOKENS,
        cache_dir=CONFIG["PATHS"]["gen_cache"],
        cache_extra=f"profile:{REFERENCE_PROFILE_PROMPT_VERSION}:single",
    )

    obj = None
    try:
        parsed = _extract_json_object(raw)
        for k in REFERENCE_PROFILE_SCHEMA["required"]:
            parsed.setdefault(k, "" if k in
                ("company", "year", "industry", "industry_inference_method",
                 "executive_summary") else [])
        parsed = _coerce_to_schema_v2(parsed)
        jsonschema.validate(instance=parsed, schema=REFERENCE_PROFILE_SCHEMA)
        parsed["_meta"] = {
            "report_id": report["report_id"],
            "schema_version": REFERENCE_PROFILE_SCHEMA_VERSION,
            "prompt_version": REFERENCE_PROFILE_PROMPT_VERSION,
            "input_truncated": was_truncated,
        }
        tmp = out_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(parsed, f, indent=2, default=str)
        os.replace(tmp, out_path)
        log_event("stage1_profile", "INFO", "profile_generated", report_id=report["report_id"])
        obj = parsed
    except (json.JSONDecodeError, ValueError, jsonschema.ValidationError) as e:
        log_event("stage1_profile", "ERROR", "profile_generation_failed",
                   report_id=report["report_id"], error=str(e))
        obj = None
    finally:
        del raw
        _free_cuda_memory()

    return obj


def run_stage1(manifest: dict) -> dict:
    n_ok, n_fail = 0, 0
    for report in manifest["reports"]:
        profile = generate_reference_profile(report)
        if profile is not None:
            n_ok += 1
        else:
            n_fail += 1
        del profile
        _free_cuda_memory()
    log_event("stage1_profile", "INFO", "stage1_complete", n_ok=n_ok, n_fail=n_fail)
    return {"n_ok": n_ok, "n_fail": n_fail}


STAGE1_RESULT = run_stage1(STAGE0_MANIFEST)
print(f"Stage 1 complete: {STAGE1_RESULT['n_ok']} profiles ok, "
      f"{STAGE1_RESULT['n_fail']} failed.")

[2026-07-07T20:42:34.465100+00:00] [WARN] [stage1_profile] input_truncated_for_memory :: {'report_id': 'f7a295e3ee6a4829330571d1', 'max_tokens': 4000}
[2026-07-07T20:42:34.824033+00:00] [ERROR] [stage1_profile] profile_generation_failed :: {'report_id': 'f7a295e3ee6a4829330571d1', 'error': "Expecting ',' delimiter: line 9 column 9773 (char 10532)"}
[2026-07-07T20:42:46.381427+00:00] [WARN] [stage1_profile] input_truncated_for_memory :: {'report_id': 'ac193e3138bbe6b4f2a48967', 'max_tokens': 4000}
[2026-07-07T20:42:46.640924+00:00] [ERROR] [stage1_profile] profile_generation_failed :: {'report_id': 'ac193e3138bbe6b4f2a48967', 'error': "Expecting ',' delimiter: line 23 column 36 (char 4833)"}
[2026-07-07T20:42:50.057066+00:00] [INFO] [stage1_profile] stage1_complete :: {'n_ok': 48, 'n_fail': 2}
Stage 1 complete: 48 profiles ok, 2 failed.


In [31]:
# Stage 1.1b: Version-aware Reference Profile adapter. Any profile already
# on Drive under the old v1 (nested-object) schema remains valid ground
# truth and is NOT regenerated. Instead, every downstream consumer (Stage
# 3/5/6 metric functions) reads profiles through get_profile_field() rather
# than indexing the JSON directly, so both schema versions present a
# uniform "flat list of strings" view without touching Stage 3/5/6 code.
_V1_NESTED_FIELDS = {
    "numerical_values": ["value", "unit", "context"],
    "kpis": ["name", "value", "unit", "period"],
    "financial_metrics": ["name", "value", "unit"],
    "environmental_metrics": ["name", "value", "unit"],
    "targets": ["description", "target_value", "target_year"],
    "dates_events": ["date", "event"],
}


def _flatten_v1_item(item, key_order: list) -> str:
    if isinstance(item, str):
        return item
    if isinstance(item, dict):
        parts = [str(item.get(k, "")) for k in key_order if item.get(k)]
        return ", ".join(parts)
    return str(item)


def get_profile_field(profile: dict, field: str) -> list:
    """Uniform accessor: returns field's contents as a flat list of strings
    regardless of whether `profile` is schema v1 (nested per-item dicts) or
    v2 (flat strings already). Every Stage 3/5/6 function that currently
    does `profile.get(field, [])` directly on a structured field should be
    switched to `get_profile_field(profile, field)` for version safety.
    Scalar fields (company/year/industry/executive_summary) are unaffected
    across versions and can still be read directly.
    """
    raw = profile.get(field, [])
    if not isinstance(raw, list):
        return []
    if field in _V1_NESTED_FIELDS:
        key_order = _V1_NESTED_FIELDS[field]
        return [_flatten_v1_item(item, key_order) for item in raw if item]
    # Already-flat fields (esg_topics, entities, risks, etc.) are identical
    # in shape across v1 and v2 — pass through, coercing to str defensively.
    return [item if isinstance(item, str) else str(item) for item in raw if item]


def get_profile_schema_version(profile: dict) -> str:
    return profile.get("_meta", {}).get("schema_version", "1.0")


log_event("stage1_adapter", "INFO", "version_adapter_ready",
          nested_v1_fields=list(_V1_NESTED_FIELDS.keys()))

[2026-07-07T20:43:45.745490+00:00] [INFO] [stage1_adapter] version_adapter_ready :: {'nested_v1_fields': ['numerical_values', 'kpis', 'financial_metrics', 'environmental_metrics', 'targets', 'dates_events']}


{'ts': '2026-07-07T20:43:45.745490+00:00',
 'stage': 'stage1_adapter',
 'level': 'INFO',
 'event': 'version_adapter_ready',
 'nested_v1_fields': ['numerical_values',
  'kpis',
  'financial_metrics',
  'environmental_metrics',
  'targets',
  'dates_events']}

In [32]:
def profile_path(report_id: str) -> str:
    return os.path.join(CONFIG["PATHS"]["reference_profiles"], f"{report_id}.json")


def _free_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [33]:
# Stage 1.2: Stage 1 QA pass — a lightweight, transparent (non-LLM) sanity
# check over generated profiles, logged for auditability. This is not a
# scientific metric; it only catches degenerate outputs (empty profiles,
# suspiciously short executive summaries) so they can be flagged before
# being used as ground truth in every later stage.
def qa_check_profile(report_id: str, profile: dict) -> list:
    issues = []
    if not profile.get("company"):
        issues.append("missing_company")
    if len(profile.get("executive_summary", "")) < 30:
        issues.append("executive_summary_too_short")
    array_fields = [
        "esg_topics", "entities", "organizations", "kpis", "financial_metrics",
        "environmental_metrics", "risks", "opportunities", "commitments",
    ]
    empty_arrays = [f for f in array_fields if len(profile.get(f, [])) == 0]
    if len(empty_arrays) >= len(array_fields) // 2:
        issues.append(f"many_empty_arrays:{empty_arrays}")
    return issues


def run_stage1_qa(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        p_path = profile_path(report["report_id"])
        if not os.path.exists(p_path):
            rows.append({"report_id": report["report_id"], "company": report["company"],
                         "status": "missing", "issues": "profile_generation_failed"})
            continue
        with open(p_path, "r") as f:
            profile = json.load(f)
        issues = qa_check_profile(report["report_id"], profile)
        status = "ok" if not issues else "flagged"
        if issues:
            log_event("stage1_qa", "WARN", "profile_flagged",
                       report_id=report["report_id"], issues=issues)
        rows.append({"report_id": report["report_id"], "company": report["company"],
                     "status": status, "issues": ";".join(issues)})
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["reference_profiles"], "_stage1_qa_report.csv")
    df.to_csv(out_csv, index=False)
    log_event("stage1_qa", "INFO", "qa_report_written", path=out_csv,
              n_ok=int((df["status"] == "ok").sum()),
              n_flagged=int((df["status"] == "flagged").sum()),
              n_missing=int((df["status"] == "missing").sum()))
    return df


STAGE1_QA_DF = run_stage1_qa(STAGE0_MANIFEST)
STAGE1_QA_DF["status"].value_counts()

[2026-07-07T20:43:54.749033+00:00] [WARN] [stage1_qa] profile_flagged :: {'report_id': 'bfd1b3b0ebe8410f3424243f', 'issues': ["many_empty_arrays:['esg_topics', 'entities', 'organizations', 'kpis', 'financial_metrics', 'environmental_metrics', 'risks', 'opportunities', 'commitments']"]}
[2026-07-07T20:43:54.756530+00:00] [WARN] [stage1_qa] profile_flagged :: {'report_id': '852815f0465c6a55e522d6cf', 'issues': ['missing_company', "many_empty_arrays:['entities', 'organizations', 'kpis', 'financial_metrics']"]}
[2026-07-07T20:43:54.763779+00:00] [WARN] [stage1_qa] profile_flagged :: {'report_id': '5e6d98d4006f9eec322846dd', 'issues': ["many_empty_arrays:['entities', 'kpis', 'financial_metrics', 'environmental_metrics', 'risks', 'opportunities', 'commitments']"]}
[2026-07-07T20:43:54.775309+00:00] [WARN] [stage1_qa] profile_flagged :: {'report_id': '4772a1d980f05b15994e318d', 'issues': ['executive_summary_too_short', "many_empty_arrays:['entities', 'organizations', 'risks', 'opportunities',

,count
status,
flagged,33
ok,15
missing,2


In [34]:
# Stage 1.3: Reference Profile ranking & grounding audit. Goes beyond the
# binary ok/flagged/missing QA pass: every generated profile gets (1) a
# weighted 0-100 completeness rating — important fields (KPIs, financials,
# environmental metrics, executive summary) weigh more than low-value
# metadata (entities, dates_events, stakeholders) — and (2) a 0-100
# grounding/similarity score measuring how much of the profile's content
# can actually be found in the source report text, to catch hallucinated
# facts that "fill" a field without being real. Field-level breakdowns are
# kept alongside the aggregate scores so weak spots are visible per report,
# not just a pass/fail label.
import re

try:
    from rapidfuzz import fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "rapidfuzz"])
    try:
        from rapidfuzz import fuzz
        _HAS_RAPIDFUZZ = True
    except ImportError:
        _HAS_RAPIDFUZZ = False
        log_event("stage1_rank", "WARN", "rapidfuzz_unavailable_exact_match_only")

# --- Importance weights: how much each schema field counts toward the
# completeness rating. Reflects that KPIs/financials/env metrics/exec
# summary are the load-bearing facts for downstream stages; entities/
# dates/stakeholders are supporting context.
FIELD_WEIGHTS = {
    "kpis": 3.0,
    "financial_metrics": 3.0,
    "environmental_metrics": 3.0,
    "executive_summary": 3.0,
    "targets": 2.5,
    "esg_topics": 2.0,
    "governance_information": 2.0,
    "social_information": 2.0,
    "risks": 2.0,
    "opportunities": 2.0,
    "commitments": 2.0,
    "regulations_frameworks_standards": 1.5,
    "future_plans": 1.5,
    "numerical_values": 1.0,
    "organizations": 1.0,
    "stakeholders": 1.0,
    "entities": 0.5,
    "dates_events": 0.5,
}
SCALAR_FIELDS = ["company", "year", "industry", "industry_inference_method"]
ARRAY_SATURATION_TARGET = 4          # >= this many items = full credit for the field
EXEC_SUMMARY_SATURATION_CHARS = 150  # >= this many chars = full credit


def _get_field_items(profile: dict, field: str) -> list:
    # Route through the version-aware adapter so v1 (nested) and v2 (flat)
    # profiles are scored on the same footing.
    try:
        return get_profile_field(profile, field)
    except NameError:
        raw = profile.get(field, [])
        return raw if isinstance(raw, list) else []


def _field_completeness(profile: dict, field: str) -> float:
    if field == "executive_summary":
        length = len(profile.get("executive_summary") or "")
        return min(length / EXEC_SUMMARY_SATURATION_CHARS, 1.0)
    items = [i for i in _get_field_items(profile, field) if str(i).strip()]
    return min(len(items) / ARRAY_SATURATION_TARGET, 1.0)


def compute_completeness_breakdown(profile: dict) -> dict:
    """Per-field completeness, 0..1, for every weighted field."""
    return {f: round(_field_completeness(profile, f), 3) for f in FIELD_WEIGHTS}


def compute_completeness_rating(profile: dict, breakdown: dict = None) -> float:
    breakdown = breakdown or compute_completeness_breakdown(profile)
    total_weight = sum(FIELD_WEIGHTS.values())
    weighted_sum = sum(FIELD_WEIGHTS[f] * breakdown[f] for f in FIELD_WEIGHTS)
    array_and_summary_score = weighted_sum / total_weight

    scalar_filled = sum(1 for f in SCALAR_FIELDS if str(profile.get(f) or "").strip())
    scalar_score = scalar_filled / len(SCALAR_FIELDS)

    combined = 0.9 * array_and_summary_score + 0.1 * scalar_score
    return round(combined * 100, 1)


# --- Grounding / similarity: does the profile's content actually trace
# back to the source report, or did the model invent it?
_STOPWORDS = set("""a an the of and or to in on for with at by from as is are was
were be been being this that these those it its their his her our your my i we
you he she they them not no""".split())


def _significant_tokens(text: str) -> list:
    tokens = re.findall(r"[A-Za-z][A-Za-z0-9\-']+|\d[\d,.\%]*", text)
    return [t for t in tokens if t.lower() not in _STOPWORDS and len(t) > 2]


def _item_grounding_score(item: str, source_lower: str) -> float:
    tokens = _significant_tokens(item)
    if not tokens:
        return 1.0  # nothing substantive to check ("N/A" etc.) — don't penalize
    matched = 0
    for tok in tokens:
        tok_l = tok.lower()
        if tok_l in source_lower:
            matched += 1
        elif _HAS_RAPIDFUZZ and fuzz.partial_ratio(tok_l, source_lower) >= 85:
            matched += 1  # near-miss: rounding, units, minor formatting differences
    return matched / len(tokens)


def compute_similarity_breakdown(profile: dict, source_text: str) -> dict:
    """Per-field grounding score, 0..1 (average over that field's items)."""
    source_lower = source_text.lower()
    breakdown = {}
    for field in FIELD_WEIGHTS:
        if field == "executive_summary":
            sentences = [s for s in re.split(r"(?<=[.!?])\s+", profile.get("executive_summary") or "") if s.strip()]
            scores = [_item_grounding_score(s, source_lower) for s in sentences]
        else:
            items = [str(i) for i in _get_field_items(profile, field) if str(i).strip()]
            scores = [_item_grounding_score(i, source_lower) for i in items]
        breakdown[field] = round(sum(scores) / len(scores), 3) if scores else None
    return breakdown


def compute_similarity_rating(breakdown: dict) -> float:
    scored = [v for v in breakdown.values() if v is not None]
    if not scored:
        return 0.0
    return round((sum(scored) / len(scored)) * 100, 1)


def run_stage1_ranking(manifest: dict) -> "pd.DataFrame":
    rows = []
    field_matrix = {}  # report_id -> {"completeness": {...}, "similarity": {...}}

    for report in manifest["reports"]:
        report_id = report["report_id"]
        p_path = profile_path(report_id)

        if not os.path.exists(p_path):
            rows.append({
                "report_id": report_id, "company": report.get("company"),
                "status": "missing", "completeness_rating": 0.0,
                "similarity_score": None,
                "weakest_high_value_fields": "profile_generation_failed",
            })
            continue

        with open(p_path, "r") as f:
            profile = json.load(f)

        completeness_bd = compute_completeness_breakdown(profile)
        completeness_rating = compute_completeness_rating(profile, completeness_bd)

        source_text = ""
        et_path = report.get("extracted_text_path")
        if et_path and os.path.exists(et_path):
            with open(et_path, "r", encoding="utf-8") as f:
                source_text = f.read()
        else:
            log_event("stage1_rank", "WARN", "extracted_text_missing", report_id=report_id)

        similarity_bd = compute_similarity_breakdown(profile, source_text) if source_text else {}
        similarity_score = compute_similarity_rating(similarity_bd) if source_text else None

        # Flag the highest-weight fields that are weakest, for a quick
        # "why is this score low" explanation without opening the matrix.
        high_value_fields = ["kpis", "financial_metrics", "environmental_metrics", "executive_summary", "targets"]
        weak = sorted(high_value_fields, key=lambda f: completeness_bd.get(f, 0))[:3]
        weak = [f for f in weak if completeness_bd.get(f, 0) < 0.5]

        field_matrix[report_id] = {"completeness": completeness_bd, "similarity": similarity_bd}

        rows.append({
            "report_id": report_id,
            "company": report.get("company"),
            "status": "ok",
            "completeness_rating": completeness_rating,
            "similarity_score": similarity_score,
            "weakest_high_value_fields": ";".join(weak) if weak else "",
        })

        if similarity_score is not None and similarity_score < 60:
            log_event("stage1_rank", "WARN", "low_grounding_score",
                       report_id=report_id, similarity_score=similarity_score)

    df = pd.DataFrame(rows).sort_values("completeness_rating", ascending=False).reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)

    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "_stage1_profile_ranking.csv")
    os.makedirs(CONFIG["PATHS"]["evaluations"], exist_ok=True)
    df.to_csv(out_csv, index=False)

    out_matrix = os.path.join(CONFIG["PATHS"]["evaluations"], "_stage1_field_matrix.json")
    with open(out_matrix, "w") as f:
        json.dump(field_matrix, f, indent=2, default=str)

    log_event("stage1_rank", "INFO", "ranking_written", path=out_csv,
              matrix_path=out_matrix, n_profiles=len(df))
    return df


STAGE1_RANKING_DF = run_stage1_ranking(STAGE0_MANIFEST)
STAGE1_RANKING_DF

[2026-07-07T20:44:00.089810+00:00] [WARN] [stage1_rank] low_grounding_score :: {'report_id': '852815f0465c6a55e522d6cf', 'similarity_score': 18.7}
[2026-07-07T20:44:00.482545+00:00] [WARN] [stage1_rank] low_grounding_score :: {'report_id': '0c635a0103394c8fad9df9a1', 'similarity_score': 0.0}
[2026-07-07T20:44:01.439955+00:00] [INFO] [stage1_rank] ranking_written :: {'path': '/content/drive/MyDrive/esg_project/evaluations/_stage1_profile_ranking.csv', 'matrix_path': '/content/drive/MyDrive/esg_project/evaluations/_stage1_field_matrix.json', 'n_profiles': 50}


,rank,report_id,company,status,completeness_rating,similarity_score,weakest_high_value_fields
0,1,6393cfa97e6ba2cf1c2ba04e,lime,ok,93.3,96.2,
1,2,26aef234bc0eb9665b42be0d,aspiration,ok,92.9,98.0,
2,3,516b9befcf0750a94a6bdcb6,cropx,ok,91.3,98.2,
3,4,a9a45ce8faeb53d10d8d0dd3,biffa,ok,87.1,98.3,
4,5,9e0815c2493abcc7f40b94b7,terracycle,ok,86.9,97.7,financial_metrics
5,6,35ed2dd1c7889989991747ee,biffa,ok,84.9,94.1,financial_metrics
6,7,e033eb352ab1e802f6f91495,forestry,ok,84.1,96.1,
7,8,e06922023a2a1d3eadd5754f,hospitality,ok,83.9,96.6,financial_metrics
8,9,24aa604e213da7d351460338,google,ok,82.5,96.5,financial_metrics
9,10,7ac27c3d237e4cdbe7278c34,olio,ok,82.2,96.0,financial_metrics


In [35]:
# Stage 2.0: Shared compression utilities — token-budget enforcement is
# implemented programmatically here (not trusted to the prompt alone), used
# identically by baseline/A/B/C so ratio-tolerance logic lives in one place.
COMPRESSED_DIR = {
    "baseline": CONFIG["PATHS"]["compressed_baseline"],
    "A": CONFIG["PATHS"]["compressed_A"],
    "B": CONFIG["PATHS"]["compressed_B"],
    "C": CONFIG["PATHS"]["compressed_C"],
}


def target_token_range(n_tokens_original: int) -> tuple:
    target = n_tokens_original * CONFIG["COMPRESSION_TARGET_RATIO"]
    tol = n_tokens_original * CONFIG["COMPRESSION_TOLERANCE"]
    return max(1, int(target - tol)), int(target + tol)


def compressed_path(variant: str, report_id: str) -> str:
    return os.path.join(COMPRESSED_DIR[variant], f"{report_id}.json")


def _trim_to_token_budget(text: str, max_tokens: int) -> str:
    """Hard trim as a last resort so the tolerance constraint is NEVER
    violated in the saved artifact, even if generation overshoots. Trims at
    the token level via the canonical tokenizer, then decodes back to text."""
    _, tok = get_model_and_tokenizer()
    ids = tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    trimmed_ids = ids[:max_tokens]
    return tok.decode(trimmed_ids, skip_special_tokens=True)


def save_compression_result(variant: str, report: dict, compressed_text: str,
                             method: str, n_regen_attempts: int, extra: dict = None) -> dict:
    n_tok = count_tokens(compressed_text)
    lo, hi = target_token_range(report["n_tokens_original"])
    result = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "method": method,
        "compressed_text": compressed_text,
        "n_tokens_original": report["n_tokens_original"],
        "n_tokens_compressed": n_tok,
        "actual_ratio": n_tok / max(1, report["n_tokens_original"]),
        "target_ratio": CONFIG["COMPRESSION_TARGET_RATIO"],
        "tolerance": CONFIG["COMPRESSION_TOLERANCE"],
        "within_tolerance": lo <= n_tok <= hi,
        "n_regen_attempts": n_regen_attempts,
    }
    if extra:
        result.update(extra)
    out_path = compressed_path(variant, report["report_id"])
    tmp = out_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(result, f, indent=2, default=str)
    os.replace(tmp, out_path)
    log_event("stage2_compress", "INFO", "compression_saved",
              variant=variant, report_id=report["report_id"],
              n_tok=n_tok, target_range=(lo, hi), within_tolerance=result["within_tolerance"])
    return result


def load_or_none(variant: str, report_id: str):
    p = compressed_path(variant, report_id)
    if os.path.exists(p):
        with open(p, "r") as f:
            return json.load(f)
    return None

In [36]:
# Stage 2.0b: Restrict compression to the top 10 highest-ranked reference
# profiles (by completeness_rating from Stage 1.3), instead of all reports
# in STAGE0_MANIFEST. Reports with a missing/low-quality profile shouldn't
# be spent on compression experiments. Deterministic given STAGE1_RANKING_DF
# (no re-ranking here) and cached to disk like the Stage 1 sample.
N_COMPRESS_TOP = 10
STAGE2_MANIFEST_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage2_top_ranked.json")


def get_stage2_manifest(ranking_df: "pd.DataFrame", full_manifest: dict,
                         n: int = N_COMPRESS_TOP) -> dict:
    if os.path.exists(STAGE2_MANIFEST_PATH):
        with open(STAGE2_MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
        log_event("stage2_manifest", "INFO", "manifest_loaded_from_cache",
                   n_reports=len(manifest["reports"]))
        return manifest

    top_ids = set(
        ranking_df[ranking_df["status"] == "ok"]
        .sort_values("completeness_rating", ascending=False)
        .head(n)["report_id"]
    )
    by_id = {r["report_id"]: r for r in full_manifest["reports"]}
    selected = [by_id[rid] for rid in top_ids if rid in by_id]

    manifest = {"n_reports": len(selected), "source": "stage1_ranking_top_n",
                "n_requested": n, "reports": selected}
    with open(STAGE2_MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    log_event("stage2_manifest", "INFO", "manifest_written",
               path=STAGE2_MANIFEST_PATH, n_reports=len(selected))
    return manifest


STAGE2_MANIFEST = get_stage2_manifest(STAGE1_RANKING_DF, STAGE0_MANIFEST, N_COMPRESS_TOP)
print(f"Stage 2 restricted to {STAGE2_MANIFEST['n_reports']} top-ranked reports: "
      f"{[r['report_id'] for r in STAGE2_MANIFEST['reports']]}")

[2026-07-07T20:44:16.776681+00:00] [INFO] [stage2_manifest] manifest_loaded_from_cache :: {'n_reports': 10}
Stage 2 restricted to 10 top-ranked reports: ['d3594605ab63dd05e0033ccb', '35ed2dd1c7889989991747ee', '9e0815c2493abcc7f40b94b7', '24aa604e213da7d351460338', '7ca6d5392f0617bdb7016a6b', 'a9a45ce8faeb53d10d8d0dd3', '1281be8990496cd34587de1c', '26aef234bc0eb9665b42be0d', 'e06922023a2a1d3eadd5754f', '09455e48dd0f7a1adbe61391']


In [37]:
# Stage 2.0b: Restrict compression to the top 10 highest-ranked reference
# profiles (by completeness_rating from Stage 1.3), instead of all reports
# in STAGE0_MANIFEST. Reports with a missing/low-quality profile shouldn't
# be spent on compression experiments. Deterministic given STAGE1_RANKING_DF
# (no re-ranking here) and cached to disk like the Stage 1 sample.
N_COMPRESS_TOP = 10
STAGE2_MANIFEST_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage2_top_ranked.json")


def get_stage2_manifest(ranking_df: "pd.DataFrame", full_manifest: dict,
                         n: int = N_COMPRESS_TOP) -> dict:
    if os.path.exists(STAGE2_MANIFEST_PATH):
        with open(STAGE2_MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
        log_event("stage2_manifest", "INFO", "manifest_loaded_from_cache",
                   n_reports=len(manifest["reports"]))
        return manifest

    top_ids = set(
        ranking_df[ranking_df["status"] == "ok"]
        .sort_values("completeness_rating", ascending=False)
        .head(n)["report_id"]
    )
    by_id = {r["report_id"]: r for r in full_manifest["reports"]}
    selected = [by_id[rid] for rid in top_ids if rid in by_id]

    manifest = {"n_reports": len(selected), "source": "stage1_ranking_top_n",
                "n_requested": n, "reports": selected}
    with open(STAGE2_MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    log_event("stage2_manifest", "INFO", "manifest_written",
               path=STAGE2_MANIFEST_PATH, n_reports=len(selected))
    return manifest


STAGE2_MANIFEST = get_stage2_manifest(STAGE1_RANKING_DF, STAGE0_MANIFEST, N_COMPRESS_TOP)
print(f"Stage 2 restricted to {STAGE2_MANIFEST['n_reports']} top-ranked reports: "
      f"{[r['report_id'] for r in STAGE2_MANIFEST['reports']]}")

[2026-07-07T20:44:20.616737+00:00] [INFO] [stage2_manifest] manifest_loaded_from_cache :: {'n_reports': 10}
Stage 2 restricted to 10 top-ranked reports: ['d3594605ab63dd05e0033ccb', '35ed2dd1c7889989991747ee', '9e0815c2493abcc7f40b94b7', '24aa604e213da7d351460338', '7ca6d5392f0617bdb7016a6b', 'a9a45ce8faeb53d10d8d0dd3', '1281be8990496cd34587de1c', '26aef234bc0eb9665b42be0d', 'e06922023a2a1d3eadd5754f', '09455e48dd0f7a1adbe61391']


In [38]:
# Stage 2.1: Compression Baseline (control) — naive truncation, no LLM
# involved. Keeps the first target-ratio fraction of tokens verbatim. This
# is the floor that A/B/C must beat on every metric.
def compress_baseline(report: dict) -> dict:
    cached = load_or_none("baseline", report["report_id"])
    if cached is not None:
        return cached

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        text = f.read()

    lo, hi = target_token_range(report["n_tokens_original"])
    target_mid = (lo + hi) // 2
    truncated = _trim_to_token_budget(text, target_mid)

    return save_compression_result(
        "baseline", report, truncated,
        method="naive_first_n_tokens_truncation",
        n_regen_attempts=0,
    )


def run_stage2_baseline(manifest: dict) -> int:
    n = 0
    for report in manifest["reports"]:
        compress_baseline(report)
        n += 1
    log_event("stage2_compress", "INFO", "baseline_compression_complete", n=n)
    return n


_n_baseline = run_stage2_baseline(STAGE2_MANIFEST)
print(f"Baseline compression complete for {_n_baseline} reports.")

[2026-07-07T20:44:26.070840+00:00] [INFO] [stage2_compress] baseline_compression_complete :: {'n': 10}
Baseline compression complete for 10 reports.


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [39]:
# Stage 2.2: Compression A (simple direct prompt) and Compression B
# (few-shot). Both enforce the token budget programmatically: after each
# generation, actual token count is measured; if outside tolerance, the
# model is re-prompted with explicit feedback ("too long by N tokens" /
# "too short by N tokens") up to COMPRESSION_MAX_REGEN_ATTEMPTS times, then
# hard-trimmed as a last-resort guarantee.
#
# RAM-safety additions vs previous version:
#   - Input text is capped to COMPRESSION_MAX_INPUT_TOKENS before being
#     embedded in the prompt (uncapped input was the actual OOM driver:
#     attention memory scales with prompt length, and some of the 50
#     reports are far longer than the ~10-15 page target window).
#   - max_new_tokens is capped at 2048 regardless of how large hi*1.5
#     computes to, so a long original report can't also blow up the
#     generation-side KV-cache budget.
#   - GPU memory is explicitly freed after every generation call (each
#     regen attempt) and after every report, not just at the end.
import gc

COMPRESSION_A_PROMPT_VERSION = "v1"
COMPRESSION_B_PROMPT_VERSION = "v1"
COMPRESSION_MAX_INPUT_TOKENS = 4000

_COMPRESSION_SYSTEM_PROMPT = (
    "You compress business reports while preserving as much factual "
    "information as possible. You never add information that is not in "
    "the source text."
)

_FEW_SHOT_EXAMPLES = [
    {
        "original": (
            "GreenCorp reduced Scope 1 emissions by 18% year-over-year, from "
            "120,000 tCO2e in 2023 to 98,400 tCO2e in 2024, driven by a "
            "transition to renewable electricity at its three largest "
            "manufacturing sites. The company also announced a new supplier "
            "code of conduct requiring all Tier 1 suppliers to disclose "
            "emissions data by 2026."
        ),
        "compressed": (
            "GreenCorp: Scope 1 emissions -18% YoY (120,000 -> 98,400 tCO2e, "
            "2023-2024), driven by renewable electricity at 3 largest plants. "
            "New supplier code of conduct: Tier 1 suppliers must disclose "
            "emissions by 2026."
        ),
    },
    {
        "original": (
            "The board's Audit Committee met four times in fiscal year 2024 "
            "and oversaw the implementation of a new whistleblower hotline, "
            "which received 42 reports, 12 of which were substantiated and "
            "led to corrective action. Employee engagement scores rose from "
            "68% to 74% following the changes."
        ),
        "compressed": (
            "FY2024: Audit Committee met 4x; oversaw new whistleblower "
            "hotline (42 reports, 12 substantiated, corrective action taken). "
            "Employee engagement 68% -> 74% post-changes."
        ),
    },
]


def _truncate_for_compression_input(text: str, max_tokens: int = COMPRESSION_MAX_INPUT_TOKENS):
    """Bound the report text fed into compression prompts by TOKEN count,
    since attention memory scales with prompt length and some reports in
    the 50-report manifest exceed the target 10-15 page window."""
    _, tok = get_model_and_tokenizer()
    ids = tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text, False
    return tok.decode(ids[:max_tokens], skip_special_tokens=True), True


def _build_compression_a_prompt(text: str, target_tokens: int) -> str:
    return (f"Compress the following report to approximately {target_tokens} tokens "
            f"while preserving the most important factual content (numbers, KPIs, "
            f"commitments, risks, entities). Do not add commentary.\n\n"
            f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:")


def _build_compression_b_prompt(text: str, target_tokens: int) -> str:
    examples_block = "\n\n".join(
        f"EXAMPLE ORIGINAL:\n{ex['original']}\n\nEXAMPLE COMPRESSED:\n{ex['compressed']}"
        for ex in _FEW_SHOT_EXAMPLES
    )
    return (f"Study these examples of dense, factual report compression:\n\n"
            f"{examples_block}\n\n"
            f"Now compress the following report the same way, to approximately "
            f"{target_tokens} tokens, preserving KPIs, numbers, commitments, risks, "
            f"and named entities. Do not add commentary.\n\n"
            f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:")


def _free_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _compress_with_regen_loop(report: dict, prompt_builder, prompt_version: str,
                               variant: str, method_name: str) -> dict:
    cached = load_or_none(variant, report["report_id"])
    if cached is not None:
        return cached

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        text = f.read()

    text, was_truncated = _truncate_for_compression_input(text)
    if was_truncated:
        log_event("stage2_compress", "WARN", "input_truncated_for_memory",
                   variant=variant, report_id=report["report_id"],
                   max_tokens=COMPRESSION_MAX_INPUT_TOKENS)

    lo, hi = target_token_range(report["n_tokens_original"])
    target_mid = (lo + hi) // 2
    feedback = ""
    result_text = None
    attempt = 1

    for attempt in range(1, CONFIG["COMPRESSION_MAX_REGEN_ATTEMPTS"] + 1):
        prompt = prompt_builder(text, target_mid) + feedback
        out = generate(
            prompt, system=_COMPRESSION_SYSTEM_PROMPT,
            max_new_tokens=min(int(hi * 1.5) + 64, 2048),
            cache_dir=CONFIG["PATHS"]["gen_cache"],
            cache_extra=f"{variant}:{prompt_version}:attempt{attempt}",
        )
        n_tok = count_tokens(out)
        result_text = out  # keep last attempt in case all attempts miss
        if lo <= n_tok <= hi:
            _free_cuda_memory()
            break
        diff = n_tok - target_mid
        direction = "too long" if diff > 0 else "too short"
        feedback = (f"\n\nYour previous output was {direction} by "
                    f"{abs(diff)} tokens (target ~{target_mid} tokens, "
                    f"range {lo}-{hi}). Rewrite it to fit the target length "
                    f"while keeping the same factual content.")
        _free_cuda_memory()

    final_n_attempts = attempt
    if not (lo <= count_tokens(result_text) <= hi):
        result_text = _trim_to_token_budget(result_text, target_mid)
        log_event("stage2_compress", "WARN", "hard_trim_applied",
                   variant=variant, report_id=report["report_id"])

    return save_compression_result(
        variant, report, result_text, method=method_name,
        n_regen_attempts=final_n_attempts,
    )


def compress_A(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_a_prompt, COMPRESSION_A_PROMPT_VERSION,
        "A", "direct_prompt",
    )


def compress_B(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_b_prompt, COMPRESSION_B_PROMPT_VERSION,
        "B", "few_shot_prompt",
    )


def run_stage2_ab(manifest: dict) -> dict:
    n_a, n_b = 0, 0
    for report in manifest["reports"]:
        compress_A(report); n_a += 1
        compress_B(report); n_b += 1
        _free_cuda_memory()
    log_event("stage2_compress", "INFO", "A_B_compression_complete", n_a=n_a, n_b=n_b)
    return {"n_a": n_a, "n_b": n_b}


STAGE2_AB_RESULT = run_stage2_ab(STAGE2_MANIFEST)
print(f"Compression A/B complete: {STAGE2_AB_RESULT}")

[2026-07-07T20:44:51.954512+00:00] [INFO] [stage2_compress] A_B_compression_complete :: {'n_a': 10, 'n_b': 10}
Compression A/B complete: {'n_a': 10, 'n_b': 10}


In [40]:
# Stage 2.3: Compression C — expert structured-extraction prompting that
# explicitly anchors on the Reference Profile schema fields, instructing the
# model to preserve business/ESG-critical information categories rather than
# compressing generically. This is the strategy expected to best beat the
# baseline on KPI/entity/numerical preservation metrics (Stage 3), since it
# is schema-aware rather than purely length-driven.
#
# Reuses _compress_with_regen_loop from Stage 2.2 (input truncation via
# COMPRESSION_MAX_INPUT_TOKENS, capped max_new_tokens, per-attempt CUDA
# cleanup already apply here — no changes needed to that shared function).
COMPRESSION_C_PROMPT_VERSION = "v1"


def _build_compression_c_prompt(text: str, target_tokens: int) -> str:
    priority_fields = (
        "KPIs and financial/environmental metrics (with exact values and units), "
        "named entities and organizations, risks, opportunities, commitments and "
        "targets (with target years), governance and social information, key dates "
        "and events, and applicable regulations/frameworks/standards"
    )
    return (
        f"You are compressing an ESG/sustainability report for an investor/analyst "
        f"audience. Produce a dense compressed version of approximately "
        f"{target_tokens} tokens that PRIORITIZES preserving, in order of "
        f"importance:\n"
        f"1. {priority_fields}\n"
        f"2. Executive-summary-level narrative context, only as space allows.\n\n"
        f"Rules:\n"
        f"- Prefer terse, information-dense phrasing (e.g. 'Scope 1: -18% YoY, "
        f"120,000->98,400 tCO2e') over full sentences.\n"
        f"- Never drop a specific number, unit, date, or named entity to save space "
        f"if a vaguer sentence could be shortened instead.\n"
        f"- Do not add any information not present in the source text.\n\n"
        f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:"
    )


def compress_C(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_c_prompt, COMPRESSION_C_PROMPT_VERSION,
        "C", "expert_structured_extraction_prompt",
    )


def run_stage2_c(manifest: dict) -> int:
    n = 0
    for report in manifest["reports"]:
        compress_C(report)
        n += 1
        _free_cuda_memory()
    log_event("stage2_compress", "INFO", "C_compression_complete", n=n)
    return n


STAGE2_C_RESULT = run_stage2_c(STAGE2_MANIFEST)  # <-- confirm/replace with STAGE2_MANIFEST if that's an intentional, separately-defined manifest
print(f"Compression C complete for {STAGE2_C_RESULT} reports.")

[2026-07-07T20:44:59.326043+00:00] [INFO] [stage2_compress] C_compression_complete :: {'n': 10}
Compression C complete for 10 reports.


In [41]:
# Stage 2.4: Stage 2 wrap-up — consolidated compression summary table across
# all variants (baseline/A/B/C) for quick sanity inspection before moving to
# Stage 3 evaluation. Flags any report/variant combination still outside
# tolerance after the hard-trim safeguard (should be empty by construction,
# but checked explicitly for auditability).
def build_stage2_summary(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        for variant in ["baseline", "A", "B", "C"]:
            res = load_or_none(variant, report["report_id"])
            if res is None:
                rows.append({
                    "report_id": report["report_id"], "company": report["company"],
                    "variant": variant, "status": "missing",
                })
                continue
            rows.append({
                "report_id": report["report_id"],
                "company": report["company"],
                "variant": variant,
                "status": "ok" if res["within_tolerance"] else "OUT_OF_TOLERANCE",
                "n_tokens_original": res["n_tokens_original"],
                "n_tokens_compressed": res["n_tokens_compressed"],
                "actual_ratio": round(res["actual_ratio"], 4),
                "n_regen_attempts": res["n_regen_attempts"],
            })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage2_compression_summary.csv")
    df.to_csv(out_csv, index=False)

    n_bad = int((df["status"] == "OUT_OF_TOLERANCE").sum())
    n_missing = int((df["status"] == "missing").sum())
    log_event("stage2_compress", "INFO" if (n_bad == 0 and n_missing == 0) else "WARN",
              "stage2_summary_written", path=out_csv,
              n_out_of_tolerance=n_bad, n_missing=n_missing, n_rows=len(df))
    return df


STAGE2_SUMMARY_DF = build_stage2_summary(STAGE2_MANIFEST)
print(f"Stage 2 complete. Summary rows: {len(STAGE2_SUMMARY_DF)}")
STAGE2_SUMMARY_DF.groupby("variant")["actual_ratio"].describe()

[2026-07-07T20:44:59.772877+00:00] [WARN] [stage2_compress] stage2_summary_written :: {'path': '/content/drive/MyDrive/esg_project/evaluations/stage2_compression_summary.csv', 'n_out_of_tolerance': 8, 'n_missing': 0, 'n_rows': 40}
Stage 2 complete. Summary rows: 40


,count,mean,std,min,25%,50%,75%,max
variant,,,,,,,,
A,10.0,0.09711,0.020152,0.0670,0.079125,0.09945,0.114225,0.1245
B,10.0,0.07676,0.027144,0.0208,0.061700,0.08210,0.089925,0.1155
C,10.0,0.07985,0.025021,0.0349,0.067700,0.07895,0.094725,0.1160
baseline,10.0,0.09989,0.000088,0.0998,0.099800,0.09990,0.099975,0.1000


In [42]:
import re

In [43]:
# Stage 3.0: Shared evaluation infrastructure — embedding model (separate
# from the generative LLM; a small local sentence-transformer is sufficient
# and far cheaper for embedding-based primary metrics than routing this
# through the 7B generative model), plus generic numeric/text utilities
# reused by every primary metric in Stage 3 and Stage 5.
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

_EMBED_MODEL = None
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # small, CPU-friendly,
    # frees the GPU for the generative model; sufficient quality for
    # semantic-similarity-style primary metrics at this project's scale.


def get_embed_model():
    global _EMBED_MODEL
    if _EMBED_MODEL is None:
        _EMBED_MODEL = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
    return _EMBED_MODEL


def embed_texts(texts: list) -> np.ndarray:
    model = get_embed_model()
    return model.encode(texts, convert_to_numpy=True, show_progress_bar=False,
                         normalize_embeddings=True)


def semantic_similarity(text_a: str, text_b: str) -> float:
    """Cosine similarity in [0,1] (embeddings are L2-normalized, so raw
    cosine in [-1,1] is rescaled to [0,1] for consistent metric range)."""
    if not text_a.strip() or not text_b.strip():
        return 0.0
    emb = embed_texts([text_a, text_b])
    cos = float(cosine_similarity(emb[0:1], emb[1:2])[0][0])
    return (cos + 1.0) / 2.0


_NUM_RE = re.compile(
    r"[-+]?\d{1,3}(?:[,.\s]\d{3})*(?:\.\d+)?%?|\d+\.\d+%?"
)


def extract_numeric_tokens(text: str) -> list:
    """Extract normalized numeric substrings (figures, percentages) from
    free text for numerical-accuracy comparisons. Normalization strips
    thousands separators so '1,200' and '1200' match."""
    raw = _NUM_RE.findall(text or "")
    normalized = []
    for r in raw:
        n = r.replace(",", "").replace(" ", "")
        normalized.append(n)
    return normalized


def set_overlap_ratio(reference_items: list, candidate_text: str,
                       fuzzy_threshold: int = 85) -> tuple:
    """Fuzzy-match each reference item (e.g. a KPI name, entity, topic)
    against presence in candidate_text. Returns (coverage_ratio, matched,
    missing) so failure analysis (Stage 6) can reuse the same computation
    instead of re-deriving matches from a bare score.

    Fuzzy (rapidfuzz partial_ratio) rather than exact substring matching is
    used because compression/expansion legitimately rephrases wording
    ('Scope 1 emissions' vs 'scope-1 CO2 emissions') while preserving the
    underlying fact — exact match would systematically undercount valid
    preservation.
    """
    reference_items = [str(i).strip() for i in reference_items if str(i).strip()]
    if not reference_items:
        return 1.0, [], []  # vacuously fully covered if nothing to preserve
    matched, missing = [], []
    cand_lower = (candidate_text or "").lower()
    for item in reference_items:
        item_lower = item.lower()
        score = fuzz.partial_ratio(item_lower, cand_lower)
        if score >= fuzzy_threshold:
            matched.append(item)
        else:
            missing.append(item)
    return len(matched) / len(reference_items), matched, missing


log_event("stage3_setup", "INFO", "eval_infra_ready", embed_model=EMBED_MODEL_NAME)

[2026-07-07T20:44:59.835604+00:00] [INFO] [stage3_setup] eval_infra_ready :: {'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'}


{'ts': '2026-07-07T20:44:59.835604+00:00',
 'stage': 'stage3_setup',
 'level': 'INFO',
 'event': 'eval_infra_ready',
 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'}

In [48]:
# Stage 3.1: Primary (transparent, rule-based / embedding-based) metrics.
# Each function returns a dict with: score (normalized 0-1 unless noted),
# and enough auxiliary detail (matched/missing lists, raw counts) to make
# the score interpretable and reusable by Stage 6 failure analysis — no
# metric here is an opaque float with no explanation attached.
#
# For every metric: mathematical formulation, normalization, and
# interpretation are documented in the docstring per project requirements.

def metric_factual_density(text: str) -> dict:
    """Factual density = (count of numeric tokens + count of proper-noun-like
    capitalized multi-word spans) / total word count.
    Formulation: density = (N_numeric + N_entities_proxy) / N_words.
    Normalization: naturally in [0, ~0.5] for dense business text; we clip
    to [0,1] and do NOT rescale further, since this metric is meant to be
    compared *relatively* across variants of the SAME report, not against
    an absolute external benchmark.
    Interpretation: higher = more factual payload per word (denser, less
    padded with connective/narrative prose); a heavily narrative-ified
    expansion would show a visible drop relative to its own compressed
    source or the original.
    """
    words = re.findall(r"\S+", text or "")
    n_words = max(1, len(words))
    n_numeric = len(extract_numeric_tokens(text))
    n_entity_proxy = len(re.findall(r"\b(?:[A-Z][a-zA-Z&\-]*\s?){2,}\b", text or ""))
    density = (n_numeric + n_entity_proxy) / n_words
    return {"score": min(1.0, density), "n_numeric_tokens": n_numeric,
            "n_entity_proxy_spans": n_entity_proxy, "n_words": n_words}


def metric_readability(text: str) -> dict:
    """Readability via a simplified Flesch Reading Ease approximation using
    syllable-count heuristics (vowel-group counting; no external NLTK
    dependency needed, keeping the metric transparent and dependency-light).
    Formulation: FRE = 206.835 - 1.015*(words/sentences) - 84.6*(syllables/words).
    Normalization: raw FRE clipped to [0,100] then divided by 100 -> [0,1].
    Interpretation: higher = easier to read. Used diagnostically, not as a
    quality target in itself (a report that becomes MUCH easier to read
    than the original may indicate oversimplification / information loss,
    not necessarily improvement) — interpret jointly with coverage metrics.
    """
    sentences = re.split(r"[.!?]+", text or "")
    sentences = [s for s in sentences if s.strip()]
    n_sentences = max(1, len(sentences))
    words = re.findall(r"[A-Za-z]+", text or "")
    n_words = max(1, len(words))

    def _count_syllables(word: str) -> int:
        word = word.lower()
        groups = re.findall(r"[aeiouy]+", word)
        n = len(groups)
        if word.endswith("e") and n > 1:
            n -= 1
        return max(1, n)

    n_syllables = sum(_count_syllables(w) for w in words) if words else 1
    fre = 206.835 - 1.015 * (n_words / n_sentences) - 84.6 * (n_syllables / n_words)
    fre_clipped = max(0.0, min(100.0, fre))
    return {"score": fre_clipped / 100.0, "raw_flesch_reading_ease": fre_clipped,
            "n_sentences": n_sentences, "n_words": n_words}


def metric_structural_organization(text: str) -> dict:
    """Structural organization = presence/consistency of discourse structure
    signals: headings/numbered sections, paragraph breaks, list markers.
    Formulation: score = mean of three binary/graded sub-signals:
      (a) has_multiple_paragraphs: >=2 paragraph breaks -> 1 else 0
      (b) heading_or_list_density: (headings + bullet/numbered lines) /
          total lines, clipped to [0,1]
      (c) avg_paragraph_length_sanity: 1.0 if mean paragraph word-count is
          within [15, 300] (neither a single wall of text nor fragment
          soup), linearly penalized outside that band.
    Normalization: already in [0,1] per sub-signal; final score is their mean.
    Interpretation: higher = more legible document structure, independent
    of content correctness (paired with factual metrics, not a substitute).
    """
    paragraphs = [p for p in re.split(r"\n\s*\n", text or "") if p.strip()]
    lines = [l for l in (text or "").split("\n") if l.strip()]
    n_lines = max(1, len(lines))

    has_multi_para = 1.0 if len(paragraphs) >= 2 else 0.0

    heading_like = sum(1 for l in lines if re.match(r"^\s*(#{1,6}\s|\d+[\.\)]\s|[-*•]\s)", l))
    heading_density = min(1.0, heading_like / n_lines * 5)  # *5 rescales typical
        # sparse heading ratios (e.g. 1 heading per 20 lines is reasonable)
        # into a meaningful [0,1] range rather than always reading near-zero

    if paragraphs:
        avg_para_words = np.mean([len(re.findall(r"\S+", p)) for p in paragraphs])
    else:
        avg_para_words = 0
    if 15 <= avg_para_words <= 300:
        para_len_score = 1.0
    elif avg_para_words < 15:
        para_len_score = avg_para_words / 15.0
    else:
        para_len_score = max(0.0, 1.0 - (avg_para_words - 300) / 700.0)

    score = float(np.mean([has_multi_para, heading_density, para_len_score]))
    return {"score": score, "n_paragraphs": len(paragraphs),
            "heading_density_raw": heading_density,
            "avg_paragraph_words": float(avg_para_words)}


def metric_document_completeness(candidate_text: str, reference_profile: dict) -> dict:
    """Document completeness = fraction of Reference Profile SECTION
    categories that have at least one representative trace in the candidate
    text (as opposed to per-item coverage, which topic_coverage/kpi/entity
    metrics already measure at finer grain). This is a coarse structural
    completeness check: did the candidate at least touch every major
    information category the original report contained?
    Formulation: completeness = (# categories with >=1 fuzzy-matched item) /
    (# non-empty categories in the reference profile).
    Normalization: naturally in [0,1].
    Interpretation: a low score means entire information CATEGORIES were
    dropped (e.g. all governance info vanished), which is a more severe
    failure mode than sparse per-item loss within a category.
    """
    categories = ["esg_topics", "entities", "organizations", "risks",
                  "opportunities", "commitments", "governance_information",
                  "social_information", "future_plans",
                  "regulations_frameworks_standards"]
    non_empty = [c for c in categories if get_profile_field(reference_profile, c)]
    if not non_empty:
        return {"score": 1.0, "categories_checked": [], "categories_missing": []}
    present, missing_cats = [], []
    for c in non_empty:
        ratio, matched, _ = set_overlap_ratio(get_profile_field(reference_profile, c), candidate_text)
        if matched:
            present.append(c)
        else:
            missing_cats.append(c)
    score = len(present) / len(non_empty)
    return {"score": score, "categories_checked": non_empty,
            "categories_missing": missing_cats}


def metric_topic_coverage(candidate_text: str, reference_profile: dict) -> dict:
    """Topic coverage = fuzzy set-overlap ratio of Reference Profile
    esg_topics found (by substring/paraphrase match) in the candidate text.
    Formulation/Normalization: see set_overlap_ratio (fraction in [0,1]).
    Interpretation: higher = more of the original ESG topic surface area
    survived into the candidate; directly relevant to investor/analyst
    completeness of coverage."""
    ratio, matched, missing = set_overlap_ratio(
        get_profile_field(reference_profile, "esg_topics"), candidate_text)
    return {"score": ratio, "matched": matched, "missing": missing}


def metric_kpi_preservation(candidate_text: str, reference_profile: dict) -> dict:
    """KPI preservation = fraction of Reference Profile KPI *records*
    (name AND value both present) preserved in the candidate.
    v2/adapter format: each kpi is a flat string, e.g.
    "Scope 1 emissions: 98,400 tCO2e (FY2024)". We require BOTH:
      - the numeric value(s) embedded in that string appear in candidate_text
      - the surrounding descriptive text fuzzy-matches candidate_text
    so a preserved number without its label (or vice versa) still fails,
    preserving the original conjunctive (AND) intent.
    Normalization: fraction in [0,1] of KPI records satisfying both checks.
    Interpretation: this is the single most investor-relevant metric in the
    suite — a preserved KPI *name* without its value is not useful to an
    analyst, hence the conjunctive (AND) requirement rather than counting
    name/value matches independently.
    """
    kpis = get_profile_field(reference_profile, "kpis")
    if not kpis:
        return {"score": 1.0, "matched": [], "missing": []}

    cand_lower = candidate_text.lower()
    cand_numbers = set(extract_numeric_tokens(candidate_text))
    matched, missing = [], []

    for kpi_str in kpis:
        kpi_numbers = set(extract_numeric_tokens(kpi_str))
        value_ok = bool(kpi_numbers) and kpi_numbers.issubset(cand_numbers)

        name_only = re.sub(r"[\d,.\%]+", "", kpi_str).strip()
        name_ok = bool(name_only) and fuzz.partial_ratio(name_only.lower(), cand_lower) >= 85

        if name_ok and value_ok:
            matched.append(kpi_str)
        else:
            missing.append({"kpi": kpi_str, "name_matched": name_ok, "value_matched": value_ok})

    return {"score": len(matched) / len(kpis), "matched": matched, "missing": missing}


def metric_numerical_accuracy(candidate_text: str, reference_profile: dict) -> dict:
    """Numerical accuracy = fraction of ALL distinct numeric values recorded
    in the Reference Profile (numerical_values + kpis + financial_metrics +
    environmental_metrics + targets) that appear verbatim (after separator
    normalization) somewhere in the candidate text.
    Formulation: exact (normalized-string) match is used here, DELIBERATELY
    stricter than the fuzzy matching used for names/topics elsewhere,
    because a numeric value that is merely "close" (e.g. 98,400 vs 98,000)
    represents an actual factual error, not an acceptable paraphrase.
    Normalization: fraction in [0,1].
    Interpretation: the primary, non-LLM-judge proxy for hallucinated or
    dropped figures — the most safety-critical metric for financial/ESG
    reporting use cases.
    """
    ref_values = set()
    for field in ["numerical_values", "kpis", "financial_metrics",
                  "environmental_metrics", "targets"]:
        for item_str in get_profile_field(reference_profile, field):
            ref_values.update(extract_numeric_tokens(item_str))
    ref_values.discard("")
    if not ref_values:
        return {"score": 1.0, "n_checked": 0, "matched": [], "missing": []}

    cand_numbers = set(extract_numeric_tokens(candidate_text))
    matched = [v for v in ref_values if v in cand_numbers]
    missing = [v for v in ref_values if v not in cand_numbers]
    return {"score": len(matched) / len(ref_values), "n_checked": len(ref_values),
            "matched": matched, "missing": missing}


def metric_entity_preservation(candidate_text: str, reference_profile: dict) -> dict:
    """Entity preservation = fuzzy set-overlap ratio over the union of
    Reference Profile entities/organizations/stakeholders.
    Formulation/Normalization: see set_overlap_ratio.
    Interpretation: higher = named parties (companies, regulators, NGOs,
    stakeholder groups) referenced in the original remain identifiable in
    the candidate — important for traceable, auditable reporting."""
    combined = (get_profile_field(reference_profile, "entities") +
                get_profile_field(reference_profile, "organizations") +
                get_profile_field(reference_profile, "stakeholders"))
    ratio, matched, missing = set_overlap_ratio(combined, candidate_text)
    return {"score": ratio, "matched": matched, "missing": missing}


def metric_traceability(candidate_text: str, original_text: str, n_samples: int = 8) -> dict:
    """Traceability = average embedding-based semantic similarity between
    each of n_samples evenly-spaced sentence windows of the candidate and
    its single most-similar sentence window in the original, i.e. "can each
    statement in the candidate be traced back to something actually said in
    the original".
    Formulation: for each candidate window w_i, sim_i = max_j cos(w_i, o_j)
    over original windows o_j; traceability = mean_i(sim_i).
    Normalization: cosine rescaled to [0,1] as in semantic_similarity.
    Interpretation: low traceability for a SPECIFIC window (not just a low
    mean) is a strong local hallucination signal, reused directly in
    Stage 6 failure analysis to localize unsupported spans.
    """
    def _windows(text, n):
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
        if not sents:
            return []
        step = max(1, len(sents) // n)
        return [" ".join(sents[i:i + step]) for i in range(0, len(sents), step)][:n]

    cand_windows = _windows(candidate_text, n_samples)
    orig_windows = _windows(original_text, n_samples * 3)  # denser reference grid
    if not cand_windows or not orig_windows:
        return {"score": 0.0, "per_window_scores": []}

    cand_emb = embed_texts(cand_windows)
    orig_emb = embed_texts(orig_windows)
    sims = cosine_similarity(cand_emb, orig_emb)  # [n_cand, n_orig], already normalized
    per_window = ((sims.max(axis=1) + 1.0) / 2.0).tolist()
    return {"score": float(np.mean(per_window)), "per_window_scores": per_window,
            "n_cand_windows": len(cand_windows), "n_orig_windows": len(orig_windows)}


log_event("stage3_setup", "INFO", "primary_metrics_defined")

[2026-07-07T20:56:57.713625+00:00] [INFO] [stage3_setup] primary_metrics_defined :: {}


{'ts': '2026-07-07T20:56:57.713625+00:00',
 'stage': 'stage3_setup',
 'level': 'INFO',
 'event': 'primary_metrics_defined'}

In [49]:
# Stage 3.2: Secondary (LLM-judge) metrics — used ONLY where a transparent
# alternative doesn't exist (semantic judgments of hallucination /
# unsupported claims / importance preservation that pure string/embedding
# matching cannot reliably capture). Every score is accompanied by a saved
# rationale string, per project requirement, so the judgment is auditable.
LLM_JUDGE_PROMPT_VERSION = "v1"

_JUDGE_SYSTEM_PROMPT = (
    "You are a strict, skeptical fact-checking auditor comparing a SOURCE "
    "document against a CANDIDATE document derived from it. You output ONLY "
    "a single valid JSON object, no prose, no markdown fences."
)


def _build_judge_prompt(source_text: str, candidate_text: str) -> str:
    return f"""Compare the CANDIDATE document against the SOURCE document.

SOURCE:
\"\"\"
{source_text}
\"\"\"

CANDIDATE:
\"\"\"
{candidate_text}
\"\"\"

Return a JSON object with exactly these keys:
- "hallucination_rate": float in [0,1]. Fraction of CANDIDATE's factual claims
  (numbers, names, events, commitments) that are NOT supported by SOURCE.
- "hallucination_rationale": string. List (in prose) the specific unsupported
  claims found, or state "none found" with a one-sentence justification.
- "unsupported_claims_count": integer. Count of distinct unsupported claims.
- "unsupported_claims_examples": array of up to 5 short strings quoting the
  specific unsupported claims (or empty array if none).
- "importance_preservation_score": float in [0,1]. How well CANDIDATE
  preserves the MOST IMPORTANT/material information from SOURCE (weight by
  business/ESG materiality, not just any detail).
- "importance_preservation_rationale": string. Explain what material
  information was preserved and/or what material information was lost.

Return ONLY the JSON object."""


def _judge_default_on_failure() -> dict:
    return {
        "hallucination_rate": None,
        "hallucination_rationale": "JUDGE_FAILED_TO_PRODUCE_VALID_JSON",
        "unsupported_claims_count": None,
        "unsupported_claims_examples": [],
        "importance_preservation_score": None,
        "importance_preservation_rationale": "JUDGE_FAILED_TO_PRODUCE_VALID_JSON",
        "judge_valid": False,
    }


def run_llm_judge(source_text: str, candidate_text: str, cache_extra: str,
                   max_attempts: int = 2) -> dict:
    """Run the LLM-judge comparison with caching and light validation.
    Caching key includes cache_extra (report_id + variant + stage) so
    identical (source, candidate) pairs are never re-judged across resumes.
    """
    prompt = _build_judge_prompt(source_text, candidate_text)
    for attempt in range(1, max_attempts + 1):
        raw = generate(
            prompt, system=_JUDGE_SYSTEM_PROMPT, max_new_tokens=1024,
            cache_dir=CONFIG["PATHS"]["gen_cache"],
            cache_extra=f"judge:{LLM_JUDGE_PROMPT_VERSION}:{cache_extra}:attempt{attempt}",
        )
        try:
            obj = _extract_json_object(raw)
            required = {"hallucination_rate", "hallucination_rationale",
                        "unsupported_claims_count", "unsupported_claims_examples",
                        "importance_preservation_score", "importance_preservation_rationale"}
            if not required.issubset(obj.keys()):
                raise ValueError("missing_required_judge_keys")
            obj["hallucination_rate"] = float(obj["hallucination_rate"])
            obj["importance_preservation_score"] = float(obj["importance_preservation_score"])
            obj["judge_valid"] = True
            return obj
        except (json.JSONDecodeError, ValueError, TypeError) as e:
            log_event("stage3_judge", "WARN", "judge_retry",
                       cache_extra=cache_extra, attempt=attempt, error=str(e))
    log_event("stage3_judge", "ERROR", "judge_failed", cache_extra=cache_extra)
    return _judge_default_on_failure()


log_event("stage3_setup", "INFO", "llm_judge_defined")

[2026-07-07T20:57:05.662082+00:00] [INFO] [stage3_setup] llm_judge_defined :: {}


{'ts': '2026-07-07T20:57:05.662082+00:00',
 'stage': 'stage3_setup',
 'level': 'INFO',
 'event': 'llm_judge_defined'}

In [50]:
import inspect
for name in ["metric_document_completeness", "metric_topic_coverage",
             "metric_kpi_preservation", "metric_numerical_accuracy",
             "metric_entity_preservation"]:
    fn = globals().get(name)
    if fn is None:
        continue
    src = inspect.getsource(fn)
    if '.get("' in src and any(f'"{k}"' in src for k in _V1_NESTED_FIELDS):
        print(f"--- {name} needs updating ---")
        print(src)
        print()

In [ ]:
# Stage 3.3: Orchestration — evaluate every compressed report (baseline, A,
# B, C) against BOTH the Original Report and the Reference Profile, per the
# project's primary comparison strategy. Results are cached to Drive per
# (report_id, variant) so re-running after interruption skips completed
# evaluations. Output is both a single consolidated JSON store and a flat
# CSV for downstream statistics (Stage 3.4).
def stage3_eval_path(report_id: str, variant: str) -> str:
    return os.path.join(CONFIG["PATHS"]["evaluations"], f"stage3_{report_id}_{variant}.json")


def evaluate_compression(report: dict, variant: str) -> dict:
    out_path = stage3_eval_path(report["report_id"], variant)
    if os.path.exists(out_path):
        with open(out_path, "r") as f:
            return json.load(f)

    comp = load_or_none(variant, report["report_id"])
    if comp is None:
        log_event("stage3_eval", "ERROR", "missing_compression_artifact",
                   report_id=report["report_id"], variant=variant)
        return None

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        original_text = f.read()
    with open(profile_path(report["report_id"]), "r") as f:
        profile = json.load(f)

    candidate_text = comp["compressed_text"]

    result = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "stage": "compression",
        # --- Primary metrics: Original -> Compressed ---
        "factual_density": metric_factual_density(candidate_text),
        "readability": metric_readability(candidate_text),
        "structural_organization": metric_structural_organization(candidate_text),
        "semantic_similarity_to_original": {
            "score": semantic_similarity(original_text, candidate_text)},
        "traceability_to_original": metric_traceability(candidate_text, original_text),
        # --- Primary metrics: Reference Profile -> Compressed ---
        "document_completeness_vs_profile": metric_document_completeness(candidate_text, profile),
        "topic_coverage_vs_profile": metric_topic_coverage(candidate_text, profile),
        "kpi_preservation_vs_profile": metric_kpi_preservation(candidate_text, profile),
        "numerical_accuracy_vs_profile": metric_numerical_accuracy(candidate_text, profile),
        "entity_preservation_vs_profile": metric_entity_preservation(candidate_text, profile),
    }

    # --- Secondary (LLM-judge) metrics: Original -> Compressed ---
    judge = run_llm_judge(original_text, candidate_text,
                           cache_extra=f"{report['report_id']}:{variant}:compression")
    result["llm_judge_vs_original"] = judge

    tmp = out_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(result, f, indent=2, default=str)
    os.replace(tmp, out_path)
    log_event("stage3_eval", "INFO", "compression_evaluated",
              report_id=report["report_id"], variant=variant)
    return result


def run_stage3(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        for variant in ["baseline", "A", "B", "C"]:
            res = evaluate_compression(report, variant)
            if res is None:
                continue
            rows.append({
                "report_id": res["report_id"], "company": res["company"], "variant": variant,
                "factual_density": res["factual_density"]["score"],
                "readability": res["readability"]["score"],
                "structural_organization": res["structural_organization"]["score"],
                "semantic_similarity_to_original": res["semantic_similarity_to_original"]["score"],
                "traceability_to_original": res["traceability_to_original"]["score"],
                "document_completeness_vs_profile": res["document_completeness_vs_profile"]["score"],
                "topic_coverage_vs_profile": res["topic_coverage_vs_profile"]["score"],
                "kpi_preservation_vs_profile": res["kpi_preservation_vs_profile"]["score"],
                "numerical_accuracy_vs_profile": res["numerical_accuracy_vs_profile"]["score"],
                "entity_preservation_vs_profile": res["entity_preservation_vs_profile"]["score"],
                "llm_judge_hallucination_rate": res["llm_judge_vs_original"]["hallucination_rate"],
                "llm_judge_importance_preservation": res["llm_judge_vs_original"]["importance_preservation_score"],
                "llm_judge_valid": res["llm_judge_vs_original"].get("judge_valid", False),
            })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage3_compression_eval.csv")
    df.to_csv(out_csv, index=False)
    log_event("stage3_eval", "INFO", "stage3_csv_written", path=out_csv, n_rows=len(df))
    return df


STAGE3_EVAL_DF = run_stage3(STAGE2_MANIFEST)
print(f"Stage 3 complete: {len(STAGE3_EVAL_DF)} (report, variant) evaluations.")
STAGE3_EVAL_DF.groupby("variant")[
    ["kpi_preservation_vs_profile", "numerical_accuracy_vs_profile",
     "entity_preservation_vs_profile", "llm_judge_hallucination_rate"]
].mean()

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos toke

[2026-07-07T20:58:12.238651+00:00] [INFO] [stage3_eval] compression_evaluated :: {'report_id': 'd3594605ab63dd05e0033ccb', 'variant': 'baseline'}


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[2026-07-07T20:59:35.421430+00:00] [INFO] [stage3_eval] compression_evaluated :: {'report_id': 'd3594605ab63dd05e0033ccb', 'variant': 'A'}


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[2026-07-07T21:03:54.599833+00:00] [INFO] [stage3_eval] compression_evaluated :: {'report_id': '35ed2dd1c7889989991747ee', 'variant': 'baseline'}


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[2026-07-07T21:05:08.658197+00:00] [INFO] [stage3_eval] compression_evaluated :: {'report_id': '35ed2dd1c7889989991747ee', 'variant': 'A'}


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [ ]:
# Stage 3.4: Statistical methodology for compression evaluation — per-metric
# distributions (mean, std, 95% CI via t-distribution) per variant, plus
# paired statistical tests (Wilcoxon signed-rank, non-parametric and robust
# to the bounded [0,1]/non-normal nature of these scores) comparing each of
# A/B/C against the baseline on the SAME reports (paired by report_id).
from scipy import stats

STAGE3_METRIC_COLUMNS = [
    "factual_density", "readability", "structural_organization",
    "semantic_similarity_to_original", "traceability_to_original",
    "document_completeness_vs_profile", "topic_coverage_vs_profile",
    "kpi_preservation_vs_profile", "numerical_accuracy_vs_profile",
    "entity_preservation_vs_profile", "llm_judge_hallucination_rate",
    "llm_judge_importance_preservation",
]


def compute_distribution_summary(df: "pd.DataFrame", metric_cols: list,
                                  group_col: str = "variant") -> "pd.DataFrame":
    """Mean/std/95% CI (t-distribution, small-sample-appropriate given
    <=100 reports) per metric per group. CI half-width = t_{0.975,n-1} *
    (std / sqrt(n)); reported as ci_low/ci_high so results can be plotted
    with error bars directly."""
    rows = []
    for group_val, sub in df.groupby(group_col):
        for col in metric_cols:
            vals = sub[col].dropna().values
            n = len(vals)
            if n == 0:
                continue
            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if n > 1 else 0.0
            if n > 1:
                t_crit = stats.t.ppf(0.975, df=n - 1)
                half_width = t_crit * std / np.sqrt(n)
            else:
                half_width = 0.0
            rows.append({
                "group": group_val, "metric": col, "n": n, "mean": mean, "std": std,
                "ci_low": mean - half_width, "ci_high": mean + half_width,
            })
    return pd.DataFrame(rows)


def paired_test_vs_baseline(df: "pd.DataFrame", metric_cols: list,
                             baseline_label: str = "baseline",
                             treatment_labels: list = ("A", "B", "C")) -> "pd.DataFrame":
    """Wilcoxon signed-rank test (paired, non-parametric) comparing each
    treatment variant against the baseline on the SAME set of reports for
    each metric. Falls back to reporting NaN p-value if fewer than 6 paired,
    non-tied observations are available (Wilcoxon's practical minimum for a
    meaningful result) rather than emitting a misleading statistic.
    Also reports the paired mean difference and its sign, so a low p-value
    is always interpreted alongside DIRECTION (beats vs underperforms
    baseline), not just significance.
    """
    rows = []
    pivot = df.pivot_table(index="report_id", columns="variant", values=metric_cols)
    for metric in metric_cols:
        if (metric, baseline_label) not in pivot.columns:
            continue
        baseline_vals = pivot[(metric, baseline_label)]
        for treat in treatment_labels:
            if (metric, treat) not in pivot.columns:
                continue
            treat_vals = pivot[(metric, treat)]
            paired = pd.concat([baseline_vals, treat_vals], axis=1).dropna()
            paired.columns = ["baseline", "treatment"]
            diffs = paired["treatment"] - paired["baseline"]
            n_nonzero = int((diffs != 0).sum())
            if n_nonzero < 6:
                rows.append({
                    "metric": metric, "treatment": treat, "n_pairs": len(paired),
                    "mean_diff": float(diffs.mean()) if len(diffs) else np.nan,
                    "wilcoxon_stat": np.nan, "p_value": np.nan,
                    "note": "insufficient_nonzero_pairs_for_wilcoxon",
                })
                continue
            stat, p = stats.wilcoxon(paired["treatment"], paired["baseline"])
            rows.append({
                "metric": metric, "treatment": treat, "n_pairs": len(paired),
                "mean_diff": float(diffs.mean()), "wilcoxon_stat": float(stat),
                "p_value": float(p),
                "beats_baseline": bool(diffs.mean() > 0),
                "note": "ok",
            })
    return pd.DataFrame(rows)


STAGE3_DIST_SUMMARY = compute_distribution_summary(STAGE3_EVAL_DF, STAGE3_METRIC_COLUMNS)
STAGE3_DIST_SUMMARY.to_csv(
    os.path.join(CONFIG["PATHS"]["evaluations"], "stage3_distribution_summary.csv"),
    index=False)

STAGE3_STAT_TESTS = paired_test_vs_baseline(STAGE3_EVAL_DF, STAGE3_METRIC_COLUMNS)
STAGE3_STAT_TESTS.to_csv(
    os.path.join(CONFIG["PATHS"]["evaluations"], "stage3_paired_tests_vs_baseline.csv"),
    index=False)

log_event("stage3_stats", "INFO", "stage3_statistics_written",
          n_distribution_rows=len(STAGE3_DIST_SUMMARY), n_test_rows=len(STAGE3_STAT_TESTS))
print("Stage 3 statistical summary:")
STAGE3_STAT_TESTS[STAGE3_STAT_TESTS["note"] == "ok"].sort_values(["metric", "treatment"])

In [ ]:
# Stage 4.0: Expansion utilities. Expansion reconstructs a full-length report
# from a compressed variant; artifacts are stored per (variant, report_id)
# under expanded/{A,B,C,baseline}/, mirroring the Stage 2 compressed/ layout
# so downstream code can reuse the same directory-keyed access pattern.
EXPANDED_DIR = {
    "baseline": CONFIG["PATHS"]["expanded_baseline"],
    "A": CONFIG["PATHS"]["expanded_A"],
    "B": CONFIG["PATHS"]["expanded_B"],
    "C": CONFIG["PATHS"]["expanded_C"],
}

# Whether to also expand the baseline as an additional control, per the
# project spec ("baseline may optionally be expanded too"). Enabled here
# because it strengthens the Compressed->Expanded diagnostic comparison
# (Stage 5) by giving every compression strategy, including the naive
# floor, a matched expansion for apples-to-apples comparison.
EXPAND_BASELINE_TOO = True
EXPANSION_VARIANTS = ["A", "B", "C"] + (["baseline"] if EXPAND_BASELINE_TOO else [])

# Expansion target: reconstruct toward the ORIGINAL report's token count
# (not a fixed absolute number), since "expansion" here means "reconstruct
# as much of the original informational content/length as plausible from
# the compressed artifact", not hit an arbitrary size.
EXPANSION_LENGTH_TOLERANCE = 0.25  # looser than compression's 3%: expansion
    # length is a secondary concern here, not the object of measurement —
    # over-constraining it risks the model padding with filler just to hit
    # a token count, which would corrupt the hallucination/traceability
    # metrics Stage 5 actually cares about.


def expanded_path(variant: str, report_id: str) -> str:
    return os.path.join(EXPANDED_DIR[variant], f"{report_id}.json")


def load_expanded_or_none(variant: str, report_id: str):
    p = expanded_path(variant, report_id)
    if os.path.exists(p):
        with open(p, "r") as f:
            return json.load(f)
    return None


def save_expansion_result(variant: str, report: dict, expanded_text: str,
                           method: str, source_compressed_n_tokens: int) -> dict:
    n_tok = count_tokens(expanded_text)
    target = report["n_tokens_original"]
    lo = int(target * (1 - EXPANSION_LENGTH_TOLERANCE))
    hi = int(target * (1 + EXPANSION_LENGTH_TOLERANCE))
    result = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "method": method,
        "expanded_text": expanded_text,
        "n_tokens_original": target,
        "n_tokens_source_compressed": source_compressed_n_tokens,
        "n_tokens_expanded": n_tok,
        "length_ratio_vs_original": n_tok / max(1, target),
        "within_length_tolerance": lo <= n_tok <= hi,
    }
    out_path = expanded_path(variant, report["report_id"])
    tmp = out_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(result, f, indent=2, default=str)
    os.replace(tmp, out_path)
    log_event("stage4_expand", "INFO", "expansion_saved",
              variant=variant, report_id=report["report_id"], n_tok=n_tok,
              within_tolerance=result["within_length_tolerance"])
    return result


log_event("stage4_setup", "INFO", "expansion_infra_ready", variants=EXPANSION_VARIANTS)

In [ ]:
# Stage 4.1: Expansion prompt design. Explicitly instructed to minimize
# hallucination (no invented specifics beyond what the compressed source
# supports) and to maximize preservation of the compressed source's actual
# content when reconstructing full prose — the model is told it MAY expand
# telegraphic/dense phrasing into full sentences and add generic connective
# language, but must NOT introduce new facts, numbers, names, or claims.
EXPANSION_PROMPT_VERSION = "v1"
EXPANSION_MAX_NEW_TOKENS = 2048  # bounded independently of input size (the
    # compressed source is short by construction), keeping KV-cache growth
    # predictable regardless of which variant/report is being expanded

_EXPANSION_SYSTEM_PROMPT = (
    "You are reconstructing a full-length business report from a compressed "
    "summary of it. Your ONLY source of truth is the compressed text given "
    "to you. You must NOT invent, estimate, or add any fact, number, name, "
    "date, or claim that is not already present or directly implied by the "
    "compressed text. You MAY expand terse phrasing into full sentences, "
    "add standard connective/transition language, and reorganize into "
    "readable prose and paragraphs. When in doubt, prefer omission over "
    "invention."
)


def _build_expansion_prompt(compressed_text: str, target_tokens: int) -> str:
    return (
        f"Below is a compressed version of a business/ESG report. Reconstruct "
        f"a full, readable report of approximately {target_tokens} tokens that "
        f"restates every fact in the compressed text in full prose, without "
        f"adding any new facts, numbers, or claims not present below.\n\n"
        f"COMPRESSED SOURCE:\n\"\"\"\n{compressed_text}\n\"\"\"\n\n"
        f"Expanded report:"
    )


def expand_variant(report: dict, variant: str) -> dict:
    cached = load_expanded_or_none(variant, report["report_id"])
    if cached is not None:
        return cached

    comp = load_or_none(variant, report["report_id"])
    if comp is None:
        log_event("stage4_expand", "ERROR", "missing_compressed_source",
                   report_id=report["report_id"], variant=variant)
        return None

    compressed_text = comp["compressed_text"]
    target_tokens = report["n_tokens_original"]
    prompt = _build_expansion_prompt(compressed_text, target_tokens)

    raw = generate(
        prompt, system=_EXPANSION_SYSTEM_PROMPT,
        max_new_tokens=EXPANSION_MAX_NEW_TOKENS,
        cache_dir=CONFIG["PATHS"]["gen_cache"],
        cache_extra=f"expand:{variant}:{EXPANSION_PROMPT_VERSION}",
    )
    result = save_expansion_result(
        variant, report, raw.strip(),
        method="constrained_reconstruction_prompt",
        source_compressed_n_tokens=comp["n_tokens_compressed"],
    )
    del raw
    _free_cuda_memory()
    return result


def run_stage4(manifest: dict) -> dict:
    counts = {v: 0 for v in EXPANSION_VARIANTS}
    for report in manifest["reports"]:
        for variant in EXPANSION_VARIANTS:
            res = expand_variant(report, variant)
            if res is not None:
                counts[variant] += 1
        _free_cuda_memory()
    log_event("stage4_expand", "INFO", "stage4_complete", counts=counts)
    return counts


STAGE4_RESULT = run_stage4(STAGE2_MANIFEST)
print(f"Stage 4 complete: {STAGE4_RESULT}")

In [ ]:
# Stage 4.2: Stage 4 wrap-up — consolidated expansion summary (length ratio
# sanity check across variants), mirroring Stage 2.4's pattern for symmetry
# between the compression and expansion stages.
def build_stage4_summary(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        for variant in EXPANSION_VARIANTS:
            res = load_expanded_or_none(variant, report["report_id"])
            if res is None:
                rows.append({"report_id": report["report_id"], "company": report["company"],
                             "variant": variant, "status": "missing"})
                continue
            rows.append({
                "report_id": report["report_id"], "company": report["company"],
                "variant": variant,
                "status": "ok" if res["within_length_tolerance"] else "LENGTH_OUTLIER",
                "n_tokens_original": res["n_tokens_original"],
                "n_tokens_expanded": res["n_tokens_expanded"],
                "length_ratio_vs_original": round(res["length_ratio_vs_original"], 4),
            })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage4_expansion_summary.csv")
    df.to_csv(out_csv, index=False)
    n_outlier = int((df["status"] == "LENGTH_OUTLIER").sum())
    n_missing = int((df["status"] == "missing").sum())
    log_event("stage4_expand", "INFO" if (n_outlier == 0 and n_missing == 0) else "WARN",
              "stage4_summary_written", path=out_csv,
              n_length_outliers=n_outlier, n_missing=n_missing)
    return df


STAGE4_SUMMARY_DF = build_stage4_summary(STAGE2_MANIFEST)
print(f"Stage 4 complete. Summary rows: {len(STAGE4_SUMMARY_DF)}")
STAGE4_SUMMARY_DF.groupby("variant")["length_ratio_vs_original"].describe()

In [ ]:
# Stage 5.0-5.1: Expansion evaluation. Reuses every metric/judge function
# defined in Stage 3 unchanged (same evaluation contract for Original->
# Compressed and Original->Expanded, per the project's comparison strategy),
# plus adds the SECONDARY/diagnostic-only Compressed->Expanded comparison.
def stage5_eval_path(report_id: str, variant: str) -> str:
    return os.path.join(CONFIG["PATHS"]["evaluations"], f"stage5_{report_id}_{variant}.json")


def evaluate_expansion(report: dict, variant: str) -> dict:
    out_path = stage5_eval_path(report["report_id"], variant)
    if os.path.exists(out_path):
        with open(out_path, "r") as f:
            return json.load(f)

    exp = load_expanded_or_none(variant, report["report_id"])
    comp = load_or_none(variant, report["report_id"])
    if exp is None or comp is None:
        log_event("stage5_eval", "ERROR", "missing_expansion_or_compression_artifact",
                   report_id=report["report_id"], variant=variant)
        return None

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        original_text = f.read()
    with open(profile_path(report["report_id"]), "r") as f:
        profile = json.load(f)

    expanded_text = exp["expanded_text"]
    compressed_text = comp["compressed_text"]

    result = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "stage": "expansion",
        # --- Primary metrics: Original -> Expanded ---
        "factual_density": metric_factual_density(expanded_text),
        "readability": metric_readability(expanded_text),
        "structural_organization": metric_structural_organization(expanded_text),
        "semantic_similarity_to_original": {
            "score": semantic_similarity(original_text, expanded_text)},
        "traceability_to_original": metric_traceability(expanded_text, original_text),
        # --- Primary metrics: Reference Profile -> Expanded ---
        "document_completeness_vs_profile": metric_document_completeness(expanded_text, profile),
        "topic_coverage_vs_profile": metric_topic_coverage(expanded_text, profile),
        "kpi_preservation_vs_profile": metric_kpi_preservation(expanded_text, profile),
        "numerical_accuracy_vs_profile": metric_numerical_accuracy(expanded_text, profile),
        "entity_preservation_vs_profile": metric_entity_preservation(expanded_text, profile),
        # --- Secondary/diagnostic ONLY: Compressed -> Expanded ---
        # Not used for primary scientific claims; measures how much the
        # expansion step itself added/lost relative to its own immediate
        # source, isolating expansion-stage effects from compression-stage
        # effects already captured in Stage 3.
        "diagnostic_semantic_similarity_compressed_to_expanded": {
            "score": semantic_similarity(compressed_text, expanded_text)},
    }

    # --- Secondary (LLM-judge) metrics: Original -> Expanded ---
    judge = run_llm_judge(original_text, expanded_text,
                           cache_extra=f"{report['report_id']}:{variant}:expansion")
    result["llm_judge_vs_original"] = judge

    tmp = out_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(result, f, indent=2, default=str)
    os.replace(tmp, out_path)
    log_event("stage5_eval", "INFO", "expansion_evaluated",
              report_id=report["report_id"], variant=variant)
    _free_cuda_memory()
    return result


def run_stage5(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        for variant in EXPANSION_VARIANTS:
            res = evaluate_expansion(report, variant)
            if res is None:
                continue
            rows.append({
                "report_id": res["report_id"], "company": res["company"], "variant": variant,
                "factual_density": res["factual_density"]["score"],
                "readability": res["readability"]["score"],
                "structural_organization": res["structural_organization"]["score"],
                "semantic_similarity_to_original": res["semantic_similarity_to_original"]["score"],
                "traceability_to_original": res["traceability_to_original"]["score"],
                "document_completeness_vs_profile": res["document_completeness_vs_profile"]["score"],
                "topic_coverage_vs_profile": res["topic_coverage_vs_profile"]["score"],
                "kpi_preservation_vs_profile": res["kpi_preservation_vs_profile"]["score"],
                "numerical_accuracy_vs_profile": res["numerical_accuracy_vs_profile"]["score"],
                "entity_preservation_vs_profile": res["entity_preservation_vs_profile"]["score"],
                "diagnostic_compressed_to_expanded_similarity":
                    res["diagnostic_semantic_similarity_compressed_to_expanded"]["score"],
                "llm_judge_hallucination_rate": res["llm_judge_vs_original"]["hallucination_rate"],
                "llm_judge_importance_preservation": res["llm_judge_vs_original"]["importance_preservation_score"],
                "llm_judge_valid": res["llm_judge_vs_original"].get("judge_valid", False),
            })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage5_expansion_eval.csv")
    df.to_csv(out_csv, index=False)
    log_event("stage5_eval", "INFO", "stage5_csv_written", path=out_csv, n_rows=len(df))
    return df


STAGE5_EVAL_DF = run_stage5(STAGE2_MANIFEST)
print(f"Stage 5 complete: {len(STAGE5_EVAL_DF)} (report, variant) evaluations.")
STAGE5_EVAL_DF.groupby("variant")[
    ["kpi_preservation_vs_profile", "numerical_accuracy_vs_profile",
     "entity_preservation_vs_profile", "llm_judge_hallucination_rate"]
].mean()

In [ ]:
# Stage 5.2: Statistical methodology for expansion evaluation — identical
# machinery as Stage 3.4, reused verbatim on the Stage 5 dataframe. Note the
# baseline comparison here is meaningful only if EXPAND_BASELINE_TOO=True
# (Cell 23); if disabled, paired_test_vs_baseline simply yields empty rows
# for treatments lacking a baseline column, which is handled gracefully by
# the existing "if (metric, baseline_label) not in pivot.columns" guard.
STAGE5_METRIC_COLUMNS = [
    "factual_density", "readability", "structural_organization",
    "semantic_similarity_to_original", "traceability_to_original",
    "document_completeness_vs_profile", "topic_coverage_vs_profile",
    "kpi_preservation_vs_profile", "numerical_accuracy_vs_profile",
    "entity_preservation_vs_profile", "llm_judge_hallucination_rate",
    "llm_judge_importance_preservation",
]

STAGE5_DIST_SUMMARY = compute_distribution_summary(STAGE5_EVAL_DF, STAGE5_METRIC_COLUMNS)
STAGE5_DIST_SUMMARY.to_csv(
    os.path.join(CONFIG["PATHS"]["evaluations"], "stage5_distribution_summary.csv"),
    index=False)

STAGE5_STAT_TESTS = paired_test_vs_baseline(
    STAGE5_EVAL_DF, STAGE5_METRIC_COLUMNS,
    treatment_labels=[v for v in EXPANSION_VARIANTS if v != "baseline"],
)
STAGE5_STAT_TESTS.to_csv(
    os.path.join(CONFIG["PATHS"]["evaluations"], "stage5_paired_tests_vs_baseline.csv"),
    index=False)

# Compression-stage vs expansion-stage comparison: for each metric shared by
# both stages, report the mean delta (expanded - compressed) per variant, to
# see whether expansion recovers, preserves, or further erodes information
# relative to the compressed artifact it was reconstructed from.
_shared_metrics = [m for m in STAGE3_METRIC_COLUMNS if m in STAGE5_METRIC_COLUMNS]
_c = STAGE3_EVAL_DF.set_index(["report_id", "variant"])[_shared_metrics]
_e = STAGE5_EVAL_DF.set_index(["report_id", "variant"])[_shared_metrics]
_common_idx = _c.index.intersection(_e.index)
STAGE_DELTA_DF = (_e.loc[_common_idx] - _c.loc[_common_idx]).reset_index()
STAGE_DELTA_DF.to_csv(
    os.path.join(CONFIG["PATHS"]["evaluations"], "stage5_vs_stage3_delta.csv"),
    index=False)

log_event("stage5_stats", "INFO", "stage5_statistics_written",
          n_distribution_rows=len(STAGE5_DIST_SUMMARY), n_test_rows=len(STAGE5_STAT_TESTS),
          n_delta_rows=len(STAGE_DELTA_DF))
print("Stage 5 statistical summary:")
STAGE5_STAT_TESTS[STAGE5_STAT_TESTS["note"] == "ok"].sort_values(["metric", "treatment"])

In [ ]:
# Stage 6.0: Failure analysis — structured, per-(report, variant, stage)
# taxonomy of WHY quality degraded, built by combining:
#   (a) the already-computed rule-based "missing"/"matched" lists from
#       Stage 3/5 primary metrics (missing KPIs, entities, topics, numerical
#       values) — reused directly, not recomputed, so failure analysis is
#       consistent by construction with the scores already reported;
#   (b) the LLM-judge's unsupported-claims examples (hallucination signal);
#   (c) a lightweight structural-degradation check (paragraph/heading loss
#       relative to the original, from metric_structural_organization).
# Output is a flat structured JSON per artifact, suitable for later
# statistical/qualitative analysis (e.g. "which failure mode correlates
# most with low importance_preservation_score").
FAILURE_CATEGORIES = [
    "missing_kpi", "missing_numerical_value", "missing_entity",
    "missing_esg_topic", "missing_risk_or_commitment", "missing_business_category",
    "hallucinated_information", "unsupported_claim", "structural_degradation",
]


def _missing_risks_and_commitments(candidate_text: str, profile: dict) -> dict:
    """Not computed as a standalone Stage 3/5 metric (risks/commitments feed
    into document_completeness_vs_profile at the category level), so it's
    derived here at item-level granularity specifically for failure
    analysis, reusing the same fuzzy set_overlap_ratio utility."""
    combined = list(profile.get("risks", [])) + list(profile.get("commitments", []))
    ratio, matched, missing = set_overlap_ratio(combined, candidate_text)
    return {"ratio": ratio, "matched": matched, "missing": missing}


def build_failure_record(report: dict, variant: str, stage: str) -> dict:
    """stage is 'compression' or 'expansion'; pulls the corresponding
    Stage 3/5 evaluation JSON (already on disk) plus the corresponding
    artifact text."""
    eval_path = (stage3_eval_path(report["report_id"], variant) if stage == "compression"
                 else stage5_eval_path(report["report_id"], variant))
    if not os.path.exists(eval_path):
        return None
    with open(eval_path, "r") as f:
        eval_result = json.load(f)

    if stage == "compression":
        artifact = load_or_none(variant, report["report_id"])
        candidate_text = artifact["compressed_text"] if artifact else ""
    else:
        artifact = load_expanded_or_none(variant, report["report_id"])
        candidate_text = artifact["expanded_text"] if artifact else ""

    with open(profile_path(report["report_id"]), "r") as f:
        profile = json.load(f)

    risks_commitments = _missing_risks_and_commitments(candidate_text, profile)
    struct = eval_result["structural_organization"]
    judge = eval_result["llm_judge_vs_original"]

    failure_record = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "stage": stage,
        "failures": {
            "missing_kpi": {
                "count": len(eval_result["kpi_preservation_vs_profile"]["missing"]),
                "items": eval_result["kpi_preservation_vs_profile"]["missing"],
            },
            "missing_numerical_value": {
                "count": len(eval_result["numerical_accuracy_vs_profile"]["missing"]),
                "items": eval_result["numerical_accuracy_vs_profile"]["missing"],
            },
            "missing_entity": {
                "count": len(eval_result["entity_preservation_vs_profile"]["missing"]),
                "items": eval_result["entity_preservation_vs_profile"]["missing"],
            },
            "missing_esg_topic": {
                "count": len(eval_result["topic_coverage_vs_profile"]["missing"]),
                "items": eval_result["topic_coverage_vs_profile"]["missing"],
            },
            "missing_risk_or_commitment": {
                "count": len(risks_commitments["missing"]),
                "items": risks_commitments["missing"],
            },
            "missing_business_category": {
                "count": len(eval_result["document_completeness_vs_profile"]["categories_missing"]),
                "items": eval_result["document_completeness_vs_profile"]["categories_missing"],
            },
            "hallucinated_information": {
                "rate": judge.get("hallucination_rate"),
                "rationale": judge.get("hallucination_rationale"),
                "judge_valid": judge.get("judge_valid", False),
            },
            "unsupported_claim": {
                "count": judge.get("unsupported_claims_count"),
                "examples": judge.get("unsupported_claims_examples", []),
            },
            "structural_degradation": {
                "structural_organization_score": struct["score"],
                "avg_paragraph_words": struct["avg_paragraph_words"],
                "flagged": struct["score"] < 0.4,  # threshold documented: below
                    # 0.4 indicates at least two of the three structural
                    # sub-signals (Cell 19) are substantially compromised
            },
        },
        "overall_importance_preservation_score": judge.get("importance_preservation_score"),
        "overall_importance_preservation_rationale": judge.get("importance_preservation_rationale"),
    }
    return failure_record


def failure_record_path(report_id: str, variant: str, stage: str) -> str:
    return os.path.join(CONFIG["PATHS"]["evaluations"],
                         f"stage6_failure_{report_id}_{variant}_{stage}.json")


def run_stage6(manifest: dict) -> list:
    all_records = []
    stage_variant_pairs = [("compression", v) for v in ["baseline", "A", "B", "C"]] + \
                          [("expansion", v) for v in EXPANSION_VARIANTS]
    for report in manifest["reports"]:
        for stage, variant in stage_variant_pairs:
            out_path = failure_record_path(report["report_id"], variant, stage)
            if os.path.exists(out_path):
                with open(out_path, "r") as f:
                    rec = json.load(f)
            else:
                rec = build_failure_record(report, variant, stage)
                if rec is None:
                    continue
                tmp = out_path + ".tmp"
                with open(tmp, "w") as f:
                    json.dump(rec, f, indent=2, default=str)
                os.replace(tmp, out_path)
            all_records.append(rec)
    log_event("stage6_failure", "INFO", "stage6_complete", n_records=len(all_records))
    return all_records


STAGE6_RECORDS = run_stage6(STAGE2_MANIFEST)
print(f"Stage 6 complete: {len(STAGE6_RECORDS)} failure-analysis records.")

In [ ]:
# Stage 6.1: Failure analysis aggregation — flatten the per-record failure
# taxonomy into a tabular CSV (counts per category per report/variant/stage)
# for statistical analysis, plus a summary pivot of mean failure counts per
# (stage, variant) so the dominant failure mode per compression/expansion
# strategy is immediately visible without re-parsing nested JSON.
def flatten_failure_record(rec: dict) -> dict:
    f = rec["failures"]
    return {
        "report_id": rec["report_id"], "company": rec["company"],
        "variant": rec["variant"], "stage": rec["stage"],
        "n_missing_kpi": f["missing_kpi"]["count"],
        "n_missing_numerical_value": f["missing_numerical_value"]["count"],
        "n_missing_entity": f["missing_entity"]["count"],
        "n_missing_esg_topic": f["missing_esg_topic"]["count"],
        "n_missing_risk_or_commitment": f["missing_risk_or_commitment"]["count"],
        "n_missing_business_category": f["missing_business_category"]["count"],
        "hallucination_rate": f["hallucinated_information"]["rate"],
        "n_unsupported_claims": f["unsupported_claim"]["count"],
        "structural_degradation_flagged": f["structural_degradation"]["flagged"],
        "structural_organization_score": f["structural_degradation"]["structural_organization_score"],
        "importance_preservation_score": rec["overall_importance_preservation_score"],
    }


STAGE6_FLAT_DF = pd.DataFrame([flatten_failure_record(r) for r in STAGE6_RECORDS])
_stage6_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage6_failure_analysis.csv")
STAGE6_FLAT_DF.to_csv(_stage6_csv, index=False)

_failure_count_cols = [
    "n_missing_kpi", "n_missing_numerical_value", "n_missing_entity",
    "n_missing_esg_topic", "n_missing_risk_or_commitment",
    "n_missing_business_category", "n_unsupported_claims",
]
STAGE6_SUMMARY_PIVOT = (
    STAGE6_FLAT_DF.groupby(["stage", "variant"])[
        _failure_count_cols + ["hallucination_rate", "structural_organization_score",
                               "importance_preservation_score"]
    ].mean().round(3)
)
_stage6_summary_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage6_failure_summary.csv")
STAGE6_SUMMARY_PIVOT.to_csv(_stage6_summary_csv)

log_event("stage6_failure", "INFO", "stage6_aggregation_written",
          detail_csv=_stage6_csv, summary_csv=_stage6_summary_csv,
          n_rows=len(STAGE6_FLAT_DF))
print("Stage 6 failure analysis summary (mean counts/scores per stage x variant):")
STAGE6_SUMMARY_PIVOT